# AI, Economic Perceptions, and Consumer Sentiment Ahead of the 2026 Midterms
**FRE-GY-7871A final project, single working notebook.** Each project day adds a numbered part below.

## Part 1 (Day 1): Economic and survey baseline

What this part does:
1. Loads the provided workbook (`data/raw/Perceptual_Bias_Data.xlsx`) and validates it.
2. Downloads cross-check data (FRED) and **caches every download** in `data/cache/`, so re-running the notebook never re-downloads. Set `FORCE_REFRESH = True` to refresh on purpose.
3. Assigns each observation an **assumed publication date** and splits it into *available* vs *post-cutoff*, so nothing from the future leaks into a prediction.
4. Builds a point-in-time monthly panel (sentiment date -> latest data actually public then). GDP stays quarterly.
5. Saves clean outputs to `data/processed/`.

**Run order:** Kernel > Restart & Run All. First run downloads; later runs use the cache.

In [ ]:
from pathlib import Path
import os, re, json, hashlib, time, shutil, subprocess, datetime as dt
from urllib.parse import urlparse, urlencode
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# ---------- Project configuration ----------
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "final_project" / "data").exists():
    ROOT = ROOT / "final_project"          # notebook opened from the repo root

DATA_RAW   = ROOT / "data" / "raw"
DATA_CACHE = ROOT / "data" / "cache"
DATA_PROC  = ROOT / "data" / "processed"
for p in (DATA_RAW, DATA_CACHE, DATA_PROC):
    p.mkdir(parents=True, exist_ok=True)

WORKBOOK      = DATA_RAW / "Perceptual_Bias_Data.xlsx"
FORCE_REFRESH = False                       # True = ignore the cache and re-download
CUTOFF        = pd.Timestamp("2026-10-13")  # information cutoff for this analysis
START         = pd.Timestamp("2010-01-31")  # first monthly observation analysed
ELECTION_DAY  = pd.Timestamp("2026-11-03")
FRED_CSV_URL  = os.environ.get("FRED_CSV_URL", "https://fred.stlouisfed.org/graph/fredgraph.csv")

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)
print("Cutoff:", CUTOFF.date(), "| analysis start:", START.date(), "| election:", ELECTION_DAY.date())

### 1.1 Download cache helper
Every download goes through `fetch_cached`. A file is fetched only if it is not already in `data/cache/`. A `manifest.json` records the URL, retrieval time (UTC), size and SHA-256 hash of every cached file, which doubles as data provenance. API keys are never written to the manifest.

In [ ]:
# Browser-like headers: some servers (FRED included) stall requests that do not look like a browser
BROWSER_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept": "application/json,text/csv,text/plain,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
MANIFEST_PATH = DATA_CACHE / "manifest.json"

def _read_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    return {}

def _write_manifest(m):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2, sort_keys=True), encoding="utf-8")

_SECRETS = set()           # API keys: scrubbed from every printed or returned error message
def _scrub(text):
    for k in _SECRETS:
        text = str(text).replace(k, "***")
    return text

_FAILED_HOSTS = set()       # hosts that failed this run: later files skip them instead of waiting again
_CURL_HOSTS = set()         # hosts where requests failed but curl worked: go straight to curl

def _looks_like_html(content):
    return content[:300].lstrip().lower().startswith((b"<!doctype", b"<html"))

def _get_with_requests(url, params, timeout, retries, name):
    last = ""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=BROWSER_HEADERS)
            r.raise_for_status()
            if _looks_like_html(r.content):
                raise ValueError("got an HTML page instead of CSV (proxy/block page?)")
            return r.content, ""
        except Exception as e:
            last = _scrub(f"{type(e).__name__}: {str(e)[:100]}")
            print(f"  attempt {attempt}/{retries} failed for {name}: {last}")
            if attempt < retries:
                time.sleep(2 * attempt)
    return None, last

def _get_with_curl(full_url, name):
    """Second route: curl.exe ships with Windows 10/11 and uses the Windows network stack like a browser."""
    exe = shutil.which("curl.exe") or shutil.which("curl")
    if not exe:
        return None, "curl not found"
    try:
        res = subprocess.run([exe, "-L", "-sS", "--fail", "--max-time", "60",
                              "-A", BROWSER_HEADERS["User-Agent"], full_url],
                             capture_output=True, timeout=75)
    except Exception as e:
        return None, _scrub(f"{type(e).__name__}: {str(e)[:100]}")
    if res.returncode != 0:
        return None, _scrub(res.stderr.decode(errors="ignore").strip()[:150])
    if _looks_like_html(res.stdout):
        return None, "got an HTML page instead of CSV"
    return res.stdout, ""

def fetch_cached(name, url, params=None, suffix=".csv", timeout=(10, 30), retries=2, curl_fallback=True):
    """Return (path, status). status is 'cache', 'downloaded' or 'failed/skipped: ...'.
    Downloads only when the file is not cached (or FORCE_REFRESH is True)."""
    path = DATA_CACHE / f"{name}{suffix}"
    if path.exists() and not FORCE_REFRESH:
        print(f"[cache]      {name}")
        return path, "cache"
    host = urlparse(url).netloc
    if host in _FAILED_HOSTS:
        print(f"[skipped]    {name} ({host} already failed this run)")
        return None, "skipped: host failed earlier in this run"
    full_url = url + (("?" + urlencode(params)) if params else "")
    method, content, err = "requests", None, ""
    if host not in _CURL_HOSTS:
        content, err = _get_with_requests(url, params, timeout, retries, name)
    if content is None and not curl_fallback:
        print(f"[FAILED]     {name} ({err})")
        return None, f"failed: {err}"
    if content is None:
        print(f"  trying curl for {name} ...")
        content, err2 = _get_with_curl(full_url, name)
        if content is None:
            print(f"[FAILED]     {name} (requests: {err or 'not tried'}; curl: {err2})")
            _FAILED_HOSTS.add(host)
            return None, f"failed: {err} | {err2}"
        method = "curl"
        _CURL_HOSTS.add(host)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(content)
    tmp.replace(path)                       # atomic: no half-written cache files
    safe_params = {k: v for k, v in (params or {}).items() if "key" not in k.lower()}
    m = _read_manifest()
    m[path.name] = {
        "url": url, "params": safe_params, "method": method,
        "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "bytes": len(content), "sha256": hashlib.sha256(content).hexdigest(),
    }
    _write_manifest(m)
    print(f"[downloaded] {name} (via {method})")
    return path, "downloaded"

def find_manual_fred_file(series_id):
    """Look in data/raw for a FRED CSV you saved by hand. Accepts FRED's own download name
    (UNRATE.csv), fred_UNRATE.csv, 'UNRATE (1).csv' and 'UNRATE.csv.csv'; case-insensitive."""
    pat = re.compile(rf"(fred_)?{re.escape(series_id)}( ?\(\d+\))?(\.csv)+", re.IGNORECASE)
    hits = [p for p in DATA_RAW.iterdir() if p.is_file() and pat.fullmatch(p.name)]
    return sorted(hits)[0] if hits else None

def fred_series(series_id, allow_download=True):
    """FRED series. Order: (1) a file you saved in data/raw, (2) the download cache, (3) a new download.
    Index = period END date."""
    path = find_manual_fred_file(series_id)
    if path is not None:
        print(f"[manual]     {series_id} from data/raw/{path.name}  (no download needed)")
        m = _read_manifest()
        m[f"manual:{path.name}"] = {"source": "saved by hand from fred.stlouisfed.org", "bytes": path.stat().st_size,
                                     "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
        _write_manifest(m)
    elif not allow_download:
        print(f"[optional]   {series_id}: not found in data/raw (skipped). To add: https://fred.stlouisfed.org/series/{series_id} > Download > CSV > data/raw/")
        return None
    else:
        path, _ = fetch_cached(f"fred_{series_id}", FRED_CSV_URL, params={"id": series_id})
        if path is None:
            found = sorted(p.name for p in DATA_RAW.glob("*") if p.is_file())
            print(f"  -> could not get {series_id}. Save https://fred.stlouisfed.org/series/{series_id} "
                  f"(Download > CSV) into data/raw/ (keep FRED's name {series_id}.csv).")
            print(f"     files currently in data/raw: {found}")
            return None
    df = pd.read_csv(path)
    dates = pd.to_datetime(df.iloc[:, 0])            # FRED stamps the START of each period
    vals = pd.to_numeric(df[series_id], errors="coerce")
    s = pd.Series(vals.values, index=dates, name=series_id).dropna()
    if series_id in ("GDPC1",):                       # quarterly -> quarter end
        s.index = s.index.to_period("Q").to_timestamp(how="end").normalize()
    else:                                             # monthly -> month end
        s.index = s.index + pd.offsets.MonthEnd(0)
    s.index = s.index.astype("datetime64[ns]")
    return s

### 1.2 Load the provided workbook
The sheet holds four side-by-side blocks (`Date` + value). Dates are **period-end** dates. The loader checks the exact header text, so a changed layout fails loudly instead of silently reading the wrong column.

In [ ]:
BLOCKS = {  # key: (date column, value column, expected value-header)
    "unemployment": (0, 1,  "Unemployment"),
    "sentiment":    (3, 4,  "Consumer Sentiment"),
    "pce_yoy":      (7, 8,  "PCE YoY"),
    "gdp_qoq":      (10, 11, "Real GDP QoQ Annualized"),
}

def load_workbook_blocks(path):
    raw = pd.read_excel(path, header=None, engine="openpyxl")
    header = raw.iloc[0]
    out = {}
    for key, (dcol, vcol, label) in BLOCKS.items():
        assert header[dcol] == "Date" and header[vcol] == label, f"Unexpected header for {key}: {header[dcol]!r}, {header[vcol]!r}"
        s = raw.iloc[1:, [dcol, vcol]].dropna(how="all").copy()
        s.columns = ["period_end", "value"]
        s["period_end"] = pd.to_datetime(s["period_end"]).astype("datetime64[ns]")
        s["value"] = pd.to_numeric(s["value"], errors="raise")
        out[key] = s.sort_values("period_end").reset_index(drop=True)
    return out

wb = load_workbook_blocks(WORKBOOK)
wb_sha = hashlib.sha256(WORKBOOK.read_bytes()).hexdigest()
print("Workbook SHA-256:", wb_sha[:16], "...")

def validate(key, s, freq):
    full = pd.date_range(s.period_end.min(), s.period_end.max(), freq=freq)
    missing = full.difference(pd.DatetimeIndex(s.period_end))
    return {
        "series": key, "rows": len(s), "first": s.period_end.min().date(), "last": s.period_end.max().date(),
        "nulls": int(s.value.isna().sum()), "duplicate_dates": int(s.period_end.duplicated().sum()),
        "missing_periods": len(missing), "min": s.value.min(), "max": s.value.max(),
    }

checks = pd.DataFrame([
    validate("unemployment", wb["unemployment"], pd.offsets.MonthEnd()),
    validate("sentiment",    wb["sentiment"],    pd.offsets.MonthEnd()),
    validate("pce_yoy",      wb["pce_yoy"],      pd.offsets.MonthEnd()),
    validate("gdp_qoq",      wb["gdp_qoq"],      pd.offsets.QuarterEnd()),
])
display(checks)

print("Missing periods (gaps are NOT filled or interpolated):")
for k, f in [("unemployment", pd.offsets.MonthEnd()), ("sentiment", pd.offsets.MonthEnd()),
             ("pce_yoy", pd.offsets.MonthEnd()), ("gdp_qoq", pd.offsets.QuarterEnd())]:
    s_ = wb[k]
    gap = pd.date_range(s_.period_end.min(), s_.period_end.max(), freq=f).difference(pd.DatetimeIndex(s_.period_end))
    if len(gap):
        print(f"  {k}: {[d.strftime('%Y-%m') for d in gap]}")

**Reading the validation table:** nulls and duplicates should be 0. One known gap: **unemployment for October 2025 is missing** (consistent with the autumn 2025 federal shutdown, when BLS did not collect that month's household survey; confirm on the BLS site). It is left as a gap, never interpolated, and the point-in-time panel simply keeps using the previous release for that month. The workbook starts in 1978 (the project asks for 2010 onward), so the earlier history is kept only for model warm-up. The four series also end on **different dates**, which matters for timing (next section).

The workbook contains the **headline** Michigan index only. The assignment also asks for *Current Economic Conditions* and *Consumer Expectations* "where available" (see 1.5).

### 1.3 Publication timing: what was knowable on which date?
The workbook is a **single, latest-vintage snapshot**. It has no publication dates and no revision history. Until real-time vintages are added (optional ALFRED step in 1.6), we use an explicit, conservative **assumed availability date** for each observation:

| Series | Assumed available | Reasoning |
|---|---|---|
| Unemployment (U-3) | period end + 10 days | BLS jobs report is normally the first Friday of the next month |
| Sentiment | period end (final reading) | Final Michigan number is published near month end; the latest month is treated as **preliminary** |
| PCE inflation | period end + 30 days | BEA personal income and outlays report comes about 4 weeks after month end |
| Real GDP | quarter end + 30 days | BEA **advance** estimate; later estimates are revisions |

These lags are assumptions to verify against the BLS/BEA/Michigan release calendars. Change them in `RELEASE_RULES` and everything downstream updates.

In [ ]:
RELEASE_RULES = {
    "unemployment": {"lag_days": 10, "basis": "BLS Employment Situation (first Friday of following month, +10d buffer)"},
    "sentiment":    {"lag_days": 0,  "basis": "Michigan final reading near month end; current month = preliminary"},
    "pce_yoy":      {"lag_days": 30, "basis": "BEA Personal Income and Outlays (~4 weeks after month end)"},
    "gdp_qoq":      {"lag_days": 30, "basis": "BEA advance GDP estimate (~4 weeks after quarter end)"},
    "pce_headline_yoy": {"lag_days": 30, "basis": "Headline PCE YoY recomputed from FRED PCEPI (BEA release, ~4 weeks after month end)"},
    "pce_core_yoy":     {"lag_days": 30, "basis": "Core PCE YoY recomputed from FRED PCEPILFE (optional)"},
}

def add_availability(key, s, cutoff=CUTOFF):
    s = s.copy()
    s["available_on"] = (s.period_end + pd.to_timedelta(RELEASE_RULES[key]["lag_days"], unit="D")).astype("datetime64[ns]")
    s["status"] = np.where(s.available_on <= cutoff, "available", "post-cutoff")
    if key == "sentiment":   # a reading for the cutoff month exists in the file -> assume preliminary release
        m = (s.status == "post-cutoff") & (s.period_end.dt.to_period("M") == cutoff.to_period("M"))
        s.loc[m, "available_on"] = pd.Timestamp(cutoff).as_unit("ns")
        s.loc[m, "status"] = "preliminary-assumed"
    s["available_on"] = s["available_on"].astype("datetime64[ns]")
    return s

series = {k: add_availability(k, v) for k, v in wb.items()}

audit = pd.DataFrame([{
    "series": k,
    "last_period_in_file": s.period_end.max().date(),
    "last_value": s.value.iloc[-1],
    "assumed_available_on": s.available_on.iloc[-1].date(),
    "latest_status": s.status.iloc[-1],
    "rows_post_cutoff": int((s.status == "post-cutoff").sum()),
} for k, s in series.items()])
print(f"Information cutoff: {CUTOFF.date()}")
audit

In [ ]:
post = {k: s[s.status == "post-cutoff"][["period_end", "value", "available_on"]] for k, s in series.items() if (s.status == "post-cutoff").any()}
prelim = {k: s[s.status == "preliminary-assumed"][["period_end", "value", "available_on"]] for k, s in series.items() if (s.status == "preliminary-assumed").any()}
print("POST-CUTOFF rows (excluded from every model input and from the dashboard 'known' data):")
for k, v in post.items():
    print(f"\n{k}:"); print(v.to_string(index=False))
print("\nPRELIMINARY rows (kept, but flagged; expect revision):")
for k, v in prelim.items():
    print(f"\n{k}:"); print(v.to_string(index=False))
if not post:
    print("none")

**Leakage flag to resolve:** if the audit above shows a **Q3 2026 GDP** value in the file, that number should not be public on the cutoff date (the advance estimate normally comes out in the last days of October). It is excluded from the point-in-time data below. Please check where that value came from (nowcast? a later data pull? a typo in the date?) and what vintage the whole file represents.

### 1.4 Cross-check against FRED (cached downloads)
Recompute what the assignment defines directly from source series and compare with the provided workbook:
* PCE inflation: `100 * (PCEPI / PCEPI 12 months earlier - 1)`
* Real GDP: quarter-over-quarter **annualized** = `100 * ((Q / Q-1)^4 - 1)`, and year-over-year = `100 * (Q / Q-4 - 1)` (kept distinct)

Differences are expected where the data were revised or the workbook was pulled at a different time. The goal is to know which, not to force a match. If a download fails (no internet, site blocked), that check is skipped and the rest of the notebook still runs.

In [ ]:
fred = {sid: fred_series(sid) for sid in ["UNRATE", "UMCSENT", "PCEPI", "GDPC1"]}
fred["PCEPILFE"] = fred_series("PCEPILFE", allow_download=False)    # optional: core PCE, used only to identify the workbook's PCE column
got = [k for k, v in fred.items() if v is not None]
print("\nFRED series available this run:", got if got else "none (offline or blocked)")

def compare(provided, ref, name, tol=0.05):
    j = provided.set_index("period_end")["value"].to_frame("workbook").join(ref.rename("recomputed"), how="inner").dropna()
    if j.empty:
        return {"check": name, "n_compared": 0}
    d = (j.workbook - j.recomputed).abs()
    return {"check": name, "n_compared": len(j), "max_abs_diff": round(d.max(), 4),
            "mean_abs_diff": round(d.mean(), 4), "share_diff_gt_tol": round((d > tol).mean(), 3),
            "last_diff": round((j.workbook - j.recomputed).iloc[-1], 4)}

gdp_level = fred["GDPC1"]
fred_derived = {}
def yoy_pct(s):
    """Year-over-year % change aligned by DATE (not row position), so a missing month cannot shift the comparison."""
    return 100 * (s / s.shift(12, freq="ME") - 1)

if fred["PCEPI"] is not None:
    fred_derived["pce_headline_yoy"] = yoy_pct(fred["PCEPI"])       # the assignment's definition of PCE inflation
if fred["PCEPILFE"] is not None:
    fred_derived["pce_core_yoy"] = yoy_pct(fred["PCEPILFE"])
if gdp_level is not None:
    fred_derived["gdp_qoq_ann"] = 100 * ((gdp_level / gdp_level.shift(1)) ** 4 - 1)
    fred_derived["gdp_yoy"] = 100 * (gdp_level / gdp_level.shift(4) - 1)

rows = []
if fred["UNRATE"] is not None:  rows.append(compare(wb["unemployment"], fred["UNRATE"], "unemployment vs FRED UNRATE", 0.05))
if fred["UMCSENT"] is not None: rows.append(compare(wb["sentiment"], fred["UMCSENT"], "sentiment vs FRED UMCSENT", 0.15))
if "pce_headline_yoy" in fred_derived: rows.append(compare(wb["pce_yoy"], fred_derived["pce_headline_yoy"], "workbook PCE vs HEADLINE PCE (from PCEPI)", 0.05))
if "pce_core_yoy" in fred_derived:     rows.append(compare(wb["pce_yoy"], fred_derived["pce_core_yoy"], "workbook PCE vs CORE PCE (from PCEPILFE)", 0.05))
if "gdp_qoq_ann" in fred_derived: rows.append(compare(wb["gdp_qoq"], fred_derived["gdp_qoq_ann"], "GDP QoQ annualized vs recomputed from GDPC1", 0.15))
cross = pd.DataFrame(rows) if rows else pd.DataFrame({"note": ["No FRED data available this run; cross-checks skipped"]})
cross

### 1.4b Which PCE measure is in the workbook?
The assignment asks for **headline** PCE inflation, `100 * (PCE price index / its value 12 months earlier - 1)`. The cross-check above shows the workbook's PCE column does **not** match that definition (it is smoother and, since spring 2026, sits well below the headline measure). Candidates are core PCE (excludes food and energy) or some other transformation. This matters because households feel gasoline and grocery prices, which headline PCE includes and core PCE does not.

**Decision for the project:** the model input is `pce_headline_yoy`, recomputed from FRED's `PCEPI` exactly as the assignment defines it. The workbook's version is kept as `pce_wb_yoy` for comparison only. To test the core-PCE explanation, save FRED series `PCEPILFE` into `data/raw/` and re-run; this cell then compares both.

In [ ]:
if "pce_headline_yoy" in fred_derived:
    w = wb["pce_yoy"].set_index("period_end")["value"].rename("workbook")
    cols = {"headline (PCEPI)": fred_derived["pce_headline_yoy"]}
    if "pce_core_yoy" in fred_derived:
        cols["core (PCEPILFE)"] = fred_derived["pce_core_yoy"]
    summ, recent = [], w.to_frame()
    for name, ref in cols.items():
        j = pd.concat([w, ref.rename("ref")], axis=1, sort=True).dropna()
        summ.append({"workbook vs": name, "months": len(j), "mean_abs_diff": round((j.workbook - j.ref).abs().mean(), 3),
                     "max_abs_diff": round((j.workbook - j.ref).abs().max(), 3), "correlation": round(j.workbook.corr(j.ref), 4),
                     "std_workbook": round(j.workbook.std(), 3), "std_ref": round(j.ref.std(), 3)})
        recent[name] = ref
    display(pd.DataFrame(summ))
    print("Last 8 months (percent, year over year):")
    display(recent.dropna().tail(8).round(3))
    best = min(summ, key=lambda r: r["mean_abs_diff"])
    print(f"Closest match to the workbook column: {best['workbook vs']} (mean abs diff {best['mean_abs_diff']}).")
    if "pce_core_yoy" not in fred_derived:
        print("Core PCE (PCEPILFE.csv) not provided, so that hypothesis is untested. Only headline was compared.")

    # GDP: where do the workbook and FRED disagree?
    if "gdp_qoq_ann" in fred_derived:
        g = pd.concat([wb["gdp_qoq"].set_index("period_end")["value"].rename("workbook"), fred_derived["gdp_qoq_ann"].rename("fred")], axis=1, sort=True).dropna()
        g["abs_diff"] = (g.workbook - g.fred).abs()
        print("\nGDP QoQ annualized: mean abs difference by decade (workbook vs current FRED vintage):")
        print(g.abs_diff.groupby(g.index.year // 10 * 10).mean().round(3).to_string())
        print("Last 10 quarters max abs diff:", round(g.abs_diff.tail(10).max(), 3))
else:
    print("FRED PCEPI not available this run; PCE definition check skipped.")

**GDP note:** the workbook's recent quarters match FRED's current numbers within rounding, but older quarters (especially 2007-2009 and 2021) differ by up to about 1 point. That pattern fits a different data vintage for older history (BEA revises past growth). The effect is small next to quarter-to-quarter swings, so the workbook GDP is kept as provided and FRED is used as a cross-check. The Q3 2026 value (3.7%) is not in FRED yet, which confirms it is not an official release.

### 1.5 Michigan sub-indices and inflation expectations
The assignment wants Current Conditions and Expectations **separated from the headline index**. They come from three monthly CSVs from the Michigan Surveys of Consumers *Tables* page (kept in `data/raw/` under their original names):

| File | Columns | Meaning |
|---|---|---|
| `tbmics.csv` | `ICS_ALL` | Headline Index of Consumer Sentiment (used only to **validate** the workbook) |
| `tbmiccice.csv` | `ICC`, `ICE` | **Current Economic Conditions** index, **Consumer Expectations** index |
| `tbmpx1px5.csv` | `PX_MD`, `PX5_MD` | **Median expected inflation**, next 12 months and next 5-10 years (%) |

Michigan's files only have a month name and a year, so dates are turned into month-end dates to match the rest of the project. Before 1978 the sub-index file is not monthly, so only 1978 onward is used. If a file is missing, the notebook continues with whatever it has.

In [ ]:
def read_umich(pattern, cols):
    hits = sorted(DATA_RAW.glob(pattern))
    if not hits:
        print(f"  [missing] data/raw/{pattern}")
        return None
    df = pd.read_csv(hits[0])
    df["date"] = (pd.to_datetime(df["Month"].str.strip() + " " + df["YYYY"].astype(str), format="%B %Y")
                  + pd.offsets.MonthEnd(0)).astype("datetime64[ns]")
    df = df[df.date >= "1978-01-01"].sort_values("date")
    assert not df.date.duplicated().any(), f"duplicate months in {hits[0].name}"
    print(f"  [loaded]  {hits[0].name}: {df.date.min():%b %Y} to {df.date.max():%b %Y} ({len(df)} months)")
    return df[["date"] + cols].reset_index(drop=True)

print("Michigan files:")
um_ics = read_umich("*tbmics.csv", ["ICS_ALL"])
um_sub = read_umich("*tbmiccice.csv", ["ICC", "ICE"])
um_px  = read_umich("*tbmpx1px5.csv", ["PX_MD", "PX5_MD"])

# Validate: Michigan's own headline file vs the headline in the provided workbook
if um_ics is not None:
    v = wb["sentiment"].rename(columns={"period_end": "date"}).merge(um_ics, on="date", how="left")
    both = v.dropna(subset=["ICS_ALL"])
    only_wb = v[v.ICS_ALL.isna()].date.dt.strftime("%b %Y").tolist()
    print(f"\nWorkbook headline vs Michigan file: {len(both)} months compared, "
          f"max abs difference = {(both.value - both.ICS_ALL).abs().max():.2f}")
    print("Months in workbook but not in Michigan file:", only_wb or "none")

sub = None
for part in (um_sub, um_px):
    if part is not None:
        sub = part if sub is None else sub.merge(part, on="date", how="outer")
if sub is not None:
    sub = sub.rename(columns={"ICC": "current", "ICE": "expectations", "PX_MD": "infl_exp_1y", "PX5_MD": "infl_exp_5y"})
    sub = sub.sort_values("date").reset_index(drop=True)
    sub.to_csv(DATA_PROC / "umich_subindices_clean.csv", index=False)
    print("\nSub-index table:", sub.date.min().date(), "to", sub.date.max().date(), f"({len(sub)} rows)")
    display(sub.tail(6))
else:
    print("\nNo sub-index files found; continuing with the headline index only.")

### 1.6 Point-in-time monthly panel
For every sentiment reading we find the **latest value of each economic series that was public on that reading's date** (`pd.merge_asof`, backward). Consequences, all intended:
* Unemployment and PCE for the *same* month are usually **not yet public**, so the panel uses the previous release and records how stale it is.
* GDP is **never interpolated to months.** Each month just points at the most recent released *quarterly* figure and shows which quarter it is.
* Rows after the cutoff cannot enter, so no later GDP estimate can explain an earlier perception.

In [ ]:
def feature_frame(key, col):
    s = series[key]
    s = s[s.status == "available"][["available_on", "period_end", "value"]].sort_values("available_on")
    return s.rename(columns={"available_on": f"{col}_available_on", "period_end": f"{col}_ref_period", "value": col})

sent = series["sentiment"]
sent = sent[sent.status.isin(["available", "preliminary-assumed"]) & (sent.period_end >= START)]
panel = (sent.rename(columns={"period_end": "sentiment_period", "available_on": "pred_date", "value": "sentiment", "status": "sentiment_status"})
              .sort_values("pred_date").reset_index(drop=True))

# add the assignment-defined inflation series (recomputed from FRED) with the same availability rule
for k in ("pce_headline_yoy", "pce_core_yoy"):
    if k in fred_derived:
        d = fred_derived[k].dropna().rename("value").rename_axis("period_end").reset_index()
        d["period_end"] = d.period_end.astype("datetime64[ns]")
        series[k] = add_availability(k, d)

feature_specs = [("unemployment", "unemployment"), ("pce_yoy", "pce_wb_yoy"), ("gdp_qoq", "gdp_qoq_ann")]
feature_specs += [(k, k) for k in ("pce_headline_yoy", "pce_core_yoy") if k in series]
feature_cols = [c for _, c in feature_specs]

for key, col in feature_specs:
    f = feature_frame(key, col)
    panel = pd.merge_asof(panel, f, left_on="pred_date", right_on=f"{col}_available_on", direction="backward")

if sub is not None:
    panel = panel.merge(sub.rename(columns={"date": "sentiment_period"}), on="sentiment_period", how="left")

# staleness in months between the sentiment month and the data's reference period
for col in feature_cols:
    panel[f"{col}_lag_months"] = ((panel.sentiment_period.dt.year - panel[f"{col}_ref_period"].dt.year) * 12
                                  + (panel.sentiment_period.dt.month - panel[f"{col}_ref_period"].dt.month))

# hard leakage test: nothing in a row may be published after that row's prediction date
for col in feature_cols:
    assert (panel[f"{col}_available_on"] <= panel.pred_date).all(), f"LEAK in {col}"
print("Leakage test passed: every feature was public on or before its prediction date.")
print(f"Panel: {len(panel)} monthly rows, {panel.sentiment_period.min().date()} to {panel.sentiment_period.max().date()}")
show_cols = ["sentiment_period", "pred_date", "sentiment", "sentiment_status", "unemployment", "pce_headline_yoy",
             "pce_wb_yoy", "pce_wb_yoy_ref_period", "gdp_qoq_ann", "gdp_qoq_ann_ref_period"]
panel[[c for c in show_cols if c in panel.columns]].tail(8)

In [ ]:
print("Typical data staleness at prediction time (months behind the sentiment month):")
panel[[f"{c}_lag_months" for c in feature_cols]].describe().loc[["mean", "min", "max"]].round(2)

### 1.7 Quarterly GDP table (kept separate from the monthly panel)
Growth is shown **two ways and labelled separately**: quarter-over-quarter annualized (as provided) and year-over-year. The YoY column here is **compounded from the four annualized rates**, which is an approximation (rates are rounded to 0.1). When the FRED level series is available, the exact YoY is added next to it.

In [ ]:
gdp = series["gdp_qoq"].rename(columns={"period_end": "quarter_end", "value": "qoq_annualized_pct"}).copy()
gdp["yoy_compounded_approx_pct"] = ((1 + gdp.qoq_annualized_pct / 100) ** 0.25).rolling(4).apply(np.prod, raw=True)
gdp["yoy_compounded_approx_pct"] = (gdp["yoy_compounded_approx_pct"] - 1) * 100
if "gdp_yoy" in fred_derived:
    gdp = gdp.merge(fred_derived["gdp_yoy"].rename("yoy_fred_exact_pct").rename_axis("quarter_end").reset_index(), on="quarter_end", how="left")
num_cols = gdp.select_dtypes("number").columns
gdp_recent = gdp[gdp.quarter_end >= "2022-01-01"].copy()
gdp_recent[num_cols] = gdp_recent[num_cols].round(2)
gdp_recent

### 1.8 Overview plots (2010 to the cutoff)
Descriptive only. These plots show levels, not a causal story.

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
known = {k: s[(s.status != "post-cutoff") & (s.period_end >= START)] for k, s in series.items()}

ax[0].plot(known["sentiment"].period_end, known["sentiment"].value, color="#1f4e79")
ax[0].set_title("Michigan Index of Consumer Sentiment (headline)")
ax[1].plot(known["unemployment"].period_end, known["unemployment"].value, color="#b45f06")
ax[1].set_title("U-3 unemployment rate (%, seasonally adjusted)")
ax[2].plot(known["pce_yoy"].period_end, known["pce_yoy"].value, color="#38761d", label="workbook PCE column (as provided)")
if "pce_headline_yoy" in known:
    ax[2].plot(known["pce_headline_yoy"].period_end, known["pce_headline_yoy"].value, color="#c0392b", ls="--", lw=1.2,
               label="headline PCE recomputed from PCEPI (assignment definition)")
ax[2].legend(loc="upper left", fontsize=8)
ax[2].set_title("PCE inflation, year over year (%)")
g = known["gdp_qoq"]
ax[3].bar(g.period_end, g.value, width=70, color="#6a1b9a")
ax[3].set_title("Real GDP growth, quarter over quarter, annualized (%) - released quarters only")
ax[3].set_ylim(-12, 12)   # 2020 Q2/Q3 swings are clipped on purpose for readability

for a in ax:
    a.axvline(CUTOFF, color="gray", ls=":", lw=1)
    a.axvline(ELECTION_DAY, color="red", ls="--", lw=1)
    a.grid(alpha=0.25)
ax[0].text(CUTOFF, ax[0].get_ylim()[1], " cutoff", color="gray", va="top", ha="right")
ax[0].text(ELECTION_DAY, ax[0].get_ylim()[1], " election", color="red", va="top", ha="left")
ax[3].set_xlim(pd.Timestamp("2010-01-01"), pd.Timestamp("2027-01-31"))
fig.tight_layout()
fig.savefig(DATA_PROC / "day1_overview.png", dpi=130)
plt.show()

### 1.9 Save Day 1 outputs
These files are what Day 2 (the economic-only model) will read, so Day 2 never has to touch raw data.

In [ ]:
panel.to_csv(DATA_PROC / "panel_pointintime_monthly.csv", index=False, encoding="utf-8")
gdp.to_csv(DATA_PROC / "gdp_quarterly.csv", index=False, encoding="utf-8")

inventory = pd.concat([s.assign(series=k) for k, s in series.items()], ignore_index=True)[
    ["series", "period_end", "value", "available_on", "status"]]
inventory.to_csv(DATA_PROC / "observation_inventory.csv", index=False, encoding="utf-8")

provenance = {
    "workbook": {"file": WORKBOOK.name, "sha256": wb_sha, "vintage": "single snapshot, publication dates unknown"},
    "cutoff": str(CUTOFF.date()),
    "release_rules_assumed": RELEASE_RULES,
    "cache_manifest": _read_manifest(),
}
(DATA_PROC / "provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
print("Saved to", DATA_PROC)
for p in sorted(DATA_PROC.iterdir()):
    if p.name != ".gitkeep":
        print(f"  {p.name:36s} {p.stat().st_size/1024:8.1f} KB")

### 1.10 Day 1 summary (computed from the data)

In [ ]:
latest = panel.iloc[-1]
print(f"Sample: {panel.sentiment_period.min():%b %Y} to {panel.sentiment_period.max():%b %Y} ({len(panel)} months) | cutoff {CUTOFF.date()}")
print(f"Latest sentiment: {latest.sentiment:.1f} for {latest.sentiment_period:%b %Y} ({latest.sentiment_status})")
print(f"Latest unemployment known then: {latest.unemployment:.1f}% for {latest.unemployment_ref_period:%b %Y}")
print(f"Latest workbook PCE column known then: {latest.pce_wb_yoy:.2f}% for {latest.pce_wb_yoy_ref_period:%b %Y}")
if "pce_headline_yoy" in panel.columns:
    print(f"Latest HEADLINE PCE YoY (recomputed, model input) known then: {latest.pce_headline_yoy:.2f}% for {latest.pce_headline_yoy_ref_period:%b %Y}")
print(f"Latest real GDP known then: {latest.gdp_qoq_ann:.1f}% annualized for quarter ending {latest.gdp_qoq_ann_ref_period:%b %Y}")
n_post = sum(int((s.status == 'post-cutoff').sum()) for s in series.values())
print(f"Post-cutoff observations excluded: {n_post}")
print("Michigan sub-indices:", ("loaded: " + ", ".join(c for c in ["current", "expectations", "infl_exp_1y", "infl_exp_5y"] if c in panel.columns)) if sub is not None else "NOT loaded (headline only)")
if sub is not None:
    miss = panel[panel[["current", "expectations"]].isna().any(axis=1)].sentiment_period.dt.strftime("%b %Y").tolist()
    print("Months with a headline reading but no sub-index yet:", miss or "none")
print("FRED cross-checks:", ", ".join(got) if got else "skipped (no download)")

## Open items carried into Day 2
* Workbook PCE column is NOT headline PCE (see 1.4b). Day 2 uses `pce_headline_yoy`. Save FRED `PCEPILFE` to test whether the workbook column is core PCE, and ask the instructor where it came from.
* Confirm the source of the workbook's Q3 2026 GDP value (3.7%); it is not in FRED yet and is excluded.
* Michigan Oct 2026 sub-indices and inflation expectations are not in the downloaded files yet (the preliminary headline is only in the workbook). Re-download Michigan's tables after the final October release.
* Replace assumed release lags with real release dates (BLS/BEA calendars) and, ideally, ALFRED first-release vintages.
* Note Michigan's phone-to-web collection change as a possible measurement break (check the methodology reports for exact dates).

---
# Part 2 (Day 2): Economic-only model and the sentiment gap

**Goal.** Predict consumer sentiment from the economy alone, then measure how far actual sentiment sits from that prediction.

**Definitions (fixed before looking at results)**
* **Sentiment gap = actual sentiment minus the model's out-of-sample prediction.** A negative gap is *unexplained pessimism under this model*. It is **not** evidence of media bias or perceptual bias: it could equally come from missing economic variables, a changing relationship, or survey-method effects.
* **Out-of-sample, one step ahead, expanding window.** For each month `t`, the model is fitted only on months before `t`, using only the economic data that was public on each of those dates (Part 1 panel). It then predicts month `t`. Nothing from `t` or later is used to fit or tune.
* **Sample:** 2010 onward. The first 60 months are used only for fitting, so the first forecast is January 2015.
* **Intervals (90%):** two kinds. *Parametric* (textbook OLS prediction interval) and *empirical* (the model's own past out-of-sample errors added to the forecast). Both are checked against how often reality actually fell inside them.

**Targets:** headline sentiment, Current Economic Conditions, Consumer Expectations.

**Specifications**
| Name | Inputs | What it is |
|---|---|---|
| **M1 economics only** | unemployment, headline PCE inflation (YoY), real GDP growth (QoQ annualized, clipped at +/-10) | The assignment's baseline |
| **M2 + price level** | M1 + cumulative price change over the last 5 years | Tests the idea that *price levels*, not just inflation rates, hurt sentiment |
| **M3 + inflation expectations** | M2 + Michigan's median expected 12-month inflation | A *perception* variable from the same survey and respondents. It is not an independent economic driver, so read it as "how much of the gap is accounted for by perceived inflation" |

GDP growth is clipped to +/-10 points because the 2020 Q2/Q3 swings (-28 and +35) would otherwise dominate the fit; a robustness check below leaves it unclipped.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from scipy import stats

# ---------- Part 2 configuration ----------
MIN_TRAIN    = 60       # months of history before the first forecast (Jan 2010 + 60 months = first forecast Jan 2015)
ALPHA        = 0.10     # 90% prediction intervals
MIN_ERR      = 24       # past out-of-sample errors required before the empirical interval is used
GDP_CLIP     = 10.0     # clip GDP growth to +/- this many points inside the models
COVID_WINDOW = (pd.Timestamp("2020-03-31"), pd.Timestamp("2021-06-30"))   # used only in a robustness check
RECENT       = (pd.Timestamp("2025-10-31"), pd.Timestamp("2026-09-30"))   # last 12 COMPLETE months (all targets exist)
# Michigan phone-to-web transition, from Michigan's own announcement (sca.isr.umich.edu/files/methodtransitionannouncement2024.pdf):
# April 2024 = 75% phone / 25% web, May = 50/50, June = 25/75, July 2024 = all web. Michigan applied NO adjustment factors.
WEB_START = pd.Timestamp("2024-04-30")   # first mixed-mode month
WEB_FULL  = pd.Timestamp("2024-07-31")   # first fully web month

INFL = "pce_headline_yoy" if "pce_headline_yoy" in panel.columns else "pce_wb_yoy"
if INFL != "pce_headline_yoy":
    print("WARNING: headline PCE (from FRED PCEPI) is not available; using the workbook PCE column instead. See section 1.4b.")

df2 = panel.copy()
df2["gdp_w"] = df2["gdp_qoq_ann"].clip(-GDP_CLIP, GDP_CLIP)

have_price = ("pce_headline_yoy" in df2.columns) and (fred.get("PCEPI") is not None)
if have_price:
    # Cumulative 5-year price change, taken at the SAME reference month as the PCE inflation figure that was public then (point in time)
    price_5y = 100 * (fred["PCEPI"] / fred["PCEPI"].shift(60, freq="ME") - 1)
    df2["price_level_5y"] = df2["pce_headline_yoy_ref_period"].map(price_5y)

SPECS = {"M1 economics only": ["unemployment", INFL, "gdp_w"]}
if have_price:
    SPECS["M2 + price level"] = SPECS["M1 economics only"] + ["price_level_5y"]
    if "infl_exp_1y" in df2.columns:
        SPECS["M3 + inflation expectations"] = SPECS["M2 + price level"] + ["infl_exp_1y"]

TARGETS = {"sentiment": "Headline sentiment"}
if "current" in df2.columns:
    TARGETS["current"] = "Current conditions"
    TARGETS["expectations"] = "Expectations"

print("Targets:", list(TARGETS.values()))
for k, v in SPECS.items():
    print(f"  {k}: {v}")
print(f"Panel rows: {len(df2)} ({df2.sentiment_period.min():%b %Y} to {df2.sentiment_period.max():%b %Y}); first forecast {df2.sentiment_period.iloc[MIN_TRAIN]:%b %Y}")

### 2.1 Rolling out-of-sample forecasts
The function below is the whole method. Every forecast for month `t` is fitted on rows strictly before `t`.

In [ ]:
def oos_forecast(df, target, cols, min_train=MIN_TRAIN, exclude=None, alpha=ALPHA, min_err=MIN_ERR):
    """One-step-ahead expanding-window OLS forecasts with parametric and empirical prediction intervals."""
    d = df.reset_index(drop=True)
    rows = []
    for t in range(min_train, len(d)):
        tr = d.iloc[:t].dropna(subset=[target] + cols)             # only months BEFORE t
        if exclude is not None:
            tr = tr[~tr.sentiment_period.between(*exclude)]
        x = d.iloc[[t]][cols]
        if x.isna().to_numpy().any() or len(tr) < len(cols) + 10:
            continue                                               # inputs for month t not available: no forecast
        fit = sm.OLS(tr[target], sm.add_constant(tr[cols])).fit()
        pr = fit.get_prediction(sm.add_constant(x, has_constant="add")).summary_frame(alpha=alpha)
        rows.append((d.sentiment_period[t], d.loc[t, target], pr["mean"].iloc[0],
                     pr["obs_ci_lower"].iloc[0], pr["obs_ci_upper"].iloc[0], tr[target].mean(), len(tr)))
    o = pd.DataFrame(rows, columns=["date", "actual", "pred", "lo_param", "hi_param", "benchmark", "n_train"]).set_index("date")
    o["gap"] = o.actual - o.pred
    lo, hi = [], []
    for i in range(len(o)):                                        # empirical interval: uses only errors from EARLIER months
        past = o.gap.iloc[:i].dropna()
        if len(past) >= min_err:
            lo.append(o.pred.iloc[i] + past.quantile(alpha / 2)); hi.append(o.pred.iloc[i] + past.quantile(1 - alpha / 2))
        else:
            lo.append(np.nan); hi.append(np.nan)
    o["lo_emp"], o["hi_emp"] = lo, hi
    return o

results = {(t, m): oos_forecast(df2, t, cols) for t in TARGETS for m, cols in SPECS.items()}
oos_all = (pd.concat(results, names=["target", "model"]).reset_index())
print("Forecast rows per target/model:", {f"{TARGETS[t]} | {m}": len(o) for (t, m), o in results.items()})

### 2.2 How well does the economy explain sentiment?
* **RMSE** = typical size of a forecast miss, in index points. **Benchmark RMSE** is the miss from simply predicting the average sentiment seen so far (no economics at all).
* **Out-of-sample R-squared** = 1 - (model squared error / benchmark squared error). Above 0 means the economics beat the plain average; 1 is perfect.
* **Coverage** = share of months where actual sentiment fell inside the 90% interval. A good interval covers about 90%.
* **Gap autocorrelation** = how much one month's gap predicts the next. Near 1 means the gap is a persistent episode rather than random monthly noise, which also makes textbook intervals too narrow.

In [ ]:
def window_mean(o, start=None, end=None):
    v = o.dropna(subset=["gap"])
    if start is not None: v = v[v.index >= start]
    if end is not None:   v = v[v.index <= end]
    return v.gap.mean() if len(v) else np.nan

def metrics_row(o):
    v = o.dropna(subset=["actual"])
    e, eb = v.actual - v.pred, v.actual - v.benchmark
    ve = v.dropna(subset=["lo_emp"])
    return {"months": len(v),
            "RMSE": np.sqrt((e ** 2).mean()), "benchmark RMSE": np.sqrt((eb ** 2).mean()),
            "OOS R2": 1 - (e ** 2).sum() / (eb ** 2).sum(),
            "coverage param": ((v.actual >= v.lo_param) & (v.actual <= v.hi_param)).mean(),
            "coverage empirical": ((ve.actual >= ve.lo_emp) & (ve.actual <= ve.hi_emp)).mean(),
            "gap autocorr": v.gap.autocorr(1),
            "mean gap 2015-19": window_mean(o, None, pd.Timestamp("2019-12-31")),
            "mean gap 2022-26": window_mean(o, pd.Timestamp("2022-01-31")),
            "mean gap last 12m": window_mean(o, *RECENT)}

metrics = pd.DataFrame({(TARGETS[t], m): metrics_row(o) for (t, m), o in results.items()}).T
metrics.index.names = ["target", "model"]
metrics.round(2)

**Reading the table**
* The 2015-19 mean gap shows whether the model was roughly unbiased *before* the recent period. If it is near zero there, the large recent gaps are not just a poorly centred model.
* If coverage is far below 90%, the intervals are too narrow. The gap is a persistent episode (autocorrelation near 1), so "outside the interval" should be read as "unusual relative to this model's usual month-to-month misses", not as a precise probability.

### 2.3 What the coefficients say (description, not causation)
Full-sample fit through September 2026 (the last month with every target), with autocorrelation-robust (Newey-West, 6 lags) standard errors. These are **associations in one history**, not causal effects.

In [ ]:
est = df2[df2.sentiment_period <= "2026-09-30"]
base_cols = SPECS["M1 economics only"]
coef_rows, fit_rows = [], []
for t, label in TARGETS.items():
    tr = est.dropna(subset=[t] + base_cols)
    f = sm.OLS(tr[t], sm.add_constant(tr[base_cols])).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
    for name in f.params.index:
        coef_rows.append({"target": label, "variable": name, "coef": f.params[name], "std_err (HAC)": f.bse[name], "p_value": f.pvalues[name]})
    fit_rows.append({"target": label, "months": int(f.nobs), "in-sample R2": f.rsquared, "Durbin-Watson": durbin_watson(f.resid)})
coef_tbl = pd.DataFrame(coef_rows)
display(coef_tbl.round(3))
display(pd.DataFrame(fit_rows).round(3))
print("Durbin-Watson near 2 = no residual autocorrelation; well below 2 = strong positive autocorrelation (standard errors are only trustworthy because of the HAC correction).")

#### Is the price-level effect learned from the data, or from one episode?
M2's price-level variable is the one that most changes the story, so check *when* its coefficient appears. The table re-fits M2 using data only up to each end date. If the coefficient is near zero before 2020 and becomes large only after the 2021-22 price surge, then it is identified from a **single episode**, and anything else that changed at the same time (the pandemic aftermath, politics, survey mode) would look identical to it in this data.

In [ ]:
if "M2 + price level" in SPECS:
    cols = SPECS["M2 + price level"]
    stab = []
    for end in ["2019-12-31", "2021-12-31", "2023-12-31", "2026-09-30"]:
        tr = df2[df2.sentiment_period <= end].dropna(subset=["sentiment"] + cols)
        f = sm.OLS(tr.sentiment, sm.add_constant(tr[cols])).fit()
        stab.append({"fitted on data through": end, "months": len(tr), **{c: f.params[c] for c in cols}})
    price_stability = pd.DataFrame(stab)
    display(price_stability.round(2))
else:
    print("M2 not available (needs FRED PCEPI).")

### 2.4 Charts
Descriptive only. Dark line = actual sentiment. The shaded band is the model's 90% empirical interval.

In [ ]:
INK, GREY, BLUE, ORANGE, AQUA = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a"
XLIM = (pd.Timestamp("2015-01-01"), pd.Timestamp("2027-02-28"))

def style(ax, ylabel):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", alpha=0.25); ax.set_ylabel(ylabel, color=GREY); ax.set_xlim(*XLIM)
    ax.tick_params(colors=GREY)
    ax.axvline(CUTOFF, color=GREY, ls=":", lw=1); ax.axvline(ELECTION_DAY, color="#c0392b", ls="--", lw=1)

# ---- Figure 1: actual vs predicted (headline) ----
o1 = results[("sentiment", "M1 economics only")]
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(o1.index, o1.lo_emp, o1.hi_emp, color=BLUE, alpha=0.15, lw=0, label="M1 90% interval (empirical)")
ax.plot(o1.index, o1.pred, color=BLUE, lw=1.6, label="M1 prediction (economics only)")
if ("sentiment", "M2 + price level") in results:
    o2 = results[("sentiment", "M2 + price level")]
    ax.plot(o2.index, o2.pred, color=ORANGE, lw=1.4, label="M2 prediction (+ price level)")
ax.plot(o1.index, o1.actual, color=INK, lw=1.8, label="Actual sentiment")
style(ax, "Index of Consumer Sentiment")
ax.set_title("Actual sentiment vs. what the economy predicted (out-of-sample)", loc="left", color=INK)
ax.legend(loc="lower left", frameon=False, fontsize=9)
ax.text(CUTOFF, ax.get_ylim()[1], " cutoff", color=GREY, va="top", ha="right", fontsize=8)
ax.text(ELECTION_DAY, ax.get_ylim()[1], " election", color="#c0392b", va="top", ha="left", fontsize=8)
fig.tight_layout(); fig.savefig(DATA_PROC / "day2_actual_vs_predicted.png", dpi=130); plt.show()

# ---- Figure 2: the gap under each specification ----
fig, ax = plt.subplots(figsize=(11, 4.5))
for (t, m), colr in zip([(("sentiment", m)) for m in SPECS], [BLUE, ORANGE, AQUA]):
    o = results[(t, m)]
    ax.plot(o.index, o.gap, color=colr, lw=1.6, label=m)
    ax.annotate(f"{o.gap.dropna().iloc[-1]:.0f}", (o.gap.dropna().index[-1], o.gap.dropna().iloc[-1]), xytext=(6, 0),
                textcoords="offset points", color=INK, fontsize=9, va="center")
ax.axhline(0, color=INK, lw=0.8)
style(ax, "Gap (actual - predicted), index points")
ax.set_title("Sentiment gap, headline index: how much each model leaves unexplained", loc="left", color=INK)
ax.legend(loc="lower left", frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig(DATA_PROC / "day2_gap_by_model.png", dpi=130); plt.show()

# ---- Figure 3: current conditions vs expectations ----
if "current" in TARGETS:
    models = list(SPECS)[:2]
    fig, axes = plt.subplots(1, len(models), figsize=(12, 4.2), sharey=True)
    for ax, m in zip(np.atleast_1d(axes), models):
        for t, colr in [("current", BLUE), ("expectations", ORANGE)]:
            o = results[(t, m)]
            ax.plot(o.index, o.gap, color=colr, lw=1.6, label=TARGETS[t])
        ax.axhline(0, color=INK, lw=0.8)
        style(ax, "Gap, index points" if ax is np.atleast_1d(axes)[0] else "")
        ax.set_title(m, loc="left", color=INK, fontsize=11)
        ax.legend(loc="lower left", frameon=False, fontsize=9)
    fig.suptitle("Gap for Current Conditions and Expectations", x=0.01, ha="left", color=INK)
    fig.tight_layout(); fig.savefig(DATA_PROC / "day2_gap_current_vs_expectations.png", dpi=130); plt.show()

### 2.5 Robustness: do the conclusions depend on my modelling choices?
Headline sentiment only. Each row changes **one** choice. If the gap barely moves, the result does not hinge on that choice.

In [ ]:
def swap(cols, old, new):
    return [new if c == old else c for c in cols]

variants = [("baseline", {}, lambda c: c),
            ("drop Mar-2020 to Jun-2021 from training", {"exclude": COVID_WINDOW}, lambda c: c),
            ("GDP growth NOT clipped", {}, lambda c: swap(c, "gdp_w", "gdp_qoq_ann")),
            ("workbook PCE column instead of headline PCE", {}, lambda c: swap(c, INFL, "pce_wb_yoy")),
            ("first forecast Jan 2013 (36 months of fitting)", {"min_train": 36}, lambda c: c),
            ("first forecast Jan 2017 (84 months of fitting)", {"min_train": 84}, lambda c: c)]
rob = []
for m in list(SPECS)[:2]:
    for name, kw, fix in variants:
        o = oos_forecast(df2, "sentiment", fix(SPECS[m]), **kw)
        v = o.dropna(subset=["actual"])
        rob.append({"model": m, "variant": name, "RMSE": np.sqrt((v.gap ** 2).mean()),
                    "mean gap 2015-19": window_mean(o, None, pd.Timestamp("2019-12-31")),
                    "mean gap 2022-26": window_mean(o, pd.Timestamp("2022-01-31")),
                    "mean gap last 12m": window_mean(o, *RECENT)})
robust_tbl = pd.DataFrame(rob)
robust_tbl.round(2)

### 2.6 Diagnostics: timing of the gap, seasonality, and the survey-mode change
* **Gap by year** shows when the gap opened.
* **Seasonality:** the Michigan index is not seasonally adjusted. A one-way test asks whether the M1 gap differs systematically by calendar month.
* **Survey mode:** Michigan changed from telephone to web interviewing; the assignment asks us to assess comparability. Per Michigan's own announcement, the change was phased in: **April 2024** 75% phone / 25% web, **May** 50/50, **June** 25/75, and **July 2024** fully web, with **no adjustment factors** applied. Michigan reports a 0.97 time-series correlation between the phone and web versions of the index and advises caution with month-to-month *changes* during the transition. That is the survey's own assessment, not independent verification. Comparing the gap before and after cannot separate a survey-method effect from a real change in sentiment, because both would move the gap at the same time. It can only show whether the gap was **already large before** the change.

In [ ]:
o1 = results[("sentiment", "M1 economics only")].dropna(subset=["gap"])
by_year = pd.DataFrame({m: results[("sentiment", m)].dropna(subset=["gap"]).gap.groupby(lambda d: d.year).mean() for m in SPECS})
print("Mean gap by calendar year, headline sentiment (index points):")
display(by_year.round(1))

groups = [g.gap.values for _, g in o1.groupby(o1.index.month)]
F, p_season = stats.f_oneway(*groups)
print(f"\nSeasonality: one-way test of M1 gap by calendar month: F = {F:.2f}, p = {p_season:.3f}  (small p would suggest calendar-month effects)")

print(f"\nSurvey mode (Michigan announcement): mixed Apr-Jun 2024, fully web from Jul 2024. Mean gap by period:")
for m in SPECS:
    g = results[("sentiment", m)].dropna(subset=["gap"]).gap
    before, mixed, web = g[g.index < WEB_START], g[(g.index >= WEB_START) & (g.index < WEB_FULL)], g[g.index >= WEB_FULL]
    print(f"  {m:30s} before Apr-2024: {before.mean():6.1f} (n={len(before)})   Apr-Jun 2024: {mixed.mean():6.1f} (n={len(mixed)})   from Jul-2024: {web.mean():6.1f} (n={len(web)})")
pre_2024 = o1[(o1.index >= "2022-01-31") & (o1.index <= "2023-12-31")].gap.mean()
print(f"\nM1 mean gap Jan 2022 to Dec 2023: {pre_2024:.1f}. If this is already large, the gap opened well before the April 2024 start of the web transition, so the mode change cannot be the whole story.")

### 2.7 Save Day 2 outputs and summary

In [ ]:
oos_all.to_csv(DATA_PROC / "day2_oos_predictions.csv", index=False, encoding="utf-8")
metrics.reset_index().to_csv(DATA_PROC / "day2_model_metrics.csv", index=False, encoding="utf-8")
coef_tbl.to_csv(DATA_PROC / "day2_coefficients_full_sample.csv", index=False, encoding="utf-8")
robust_tbl.to_csv(DATA_PROC / "day2_robustness.csv", index=False, encoding="utf-8")

h = results[("sentiment", "M1 economics only")]
last = h.iloc[-1]
mr = metrics.loc[("Headline sentiment", "M1 economics only")]
print(f"DAY 2 SUMMARY (data through {CUTOFF.date()}; oldest forecast {h.index.min():%b %Y}, newest {h.index.max():%b %Y})")
print(f"- Economics-only model (M1) beats the plain-average benchmark: RMSE {mr['RMSE']:.1f} vs {mr['benchmark RMSE']:.1f} index points, out-of-sample R2 = {mr['OOS R2']:.2f}.")
print(f"- Before the recent period the model was roughly centred: mean gap 2015-19 = {mr['mean gap 2015-19']:+.1f}.")
print(f"- Since 2022 the mean gap is {mr['mean gap 2022-26']:+.1f}; over the last 12 complete months ({RECENT[0]:%b %Y} to {RECENT[1]:%b %Y}) it is {mr['mean gap last 12m']:+.1f}.")
inside = (last.actual >= last.lo_emp) and (last.actual <= last.hi_emp)
print(f"- Latest month ({h.index[-1]:%b %Y}, preliminary): actual {last.actual:.1f}, predicted {last.pred:.1f}, gap {last.gap:+.1f}; "
      f"90% empirical interval [{last.lo_emp:.1f}, {last.hi_emp:.1f}] -> actual is {'inside' if inside else 'outside'} it.")
print(f"- Interval honesty: coverage of the 90% intervals was {mr['coverage param']:.0%} (parametric) and {mr['coverage empirical']:.0%} (empirical); gap autocorrelation {mr['gap autocorr']:.2f}.")
if "M2 + price level" in SPECS:
    m2 = metrics.loc[("Headline sentiment", "M2 + price level")]
    print(f"- Adding the 5-year price level (M2) cuts the recent gap from {mr['mean gap last 12m']:+.1f} to {m2['mean gap last 12m']:+.1f} and RMSE from {mr['RMSE']:.1f} to {m2['RMSE']:.1f}, "
          f"BUT its coefficient is learned from the 2021-22 episode only (see 2.3). Treat as a hypothesis, not a finding.")
if "current" in TARGETS:
    c1, e1 = metrics.loc[("Current conditions", "M1 economics only")], metrics.loc[("Expectations", "M1 economics only")]
    print(f"- M1 recent gap by component: Current conditions {c1['mean gap last 12m']:+.1f}, Expectations {e1['mean gap last 12m']:+.1f}.")
print("- A negative gap is unexplained pessimism under this model, not proof of media or perceptual bias.")
print("\nSaved:", ", ".join(p.name for p in sorted(DATA_PROC.glob("day2_*"))))

## Open items carried into Day 3
* Survey-mode dates are now set from Michigan's announcement. For the report, use Michigan's published mode-comparison research to judge how large any method effect could be.
* Real income, interest rates and housing costs are plausible omitted economic drivers (alternative explanation 1). Only price level and inflation expectations are tested so far.
* The price-level result rests on one episode. Day 5's three-model comparison should test it with subgroup data (income, age, party) where available.
* Day 3 starts the media corpus. Search terms, date range, source list and inclusion rules must be written down **before** collecting.

---
# Part 3 (Day 3): The media corpus

**Goal.** Build a transparent, reproducible corpus of economic news and social posts that Day 4 can code for tone, framing and accuracy. This part only *collects and describes*. It does not measure anything about bias.

## 3.0 Collection protocol (fixed before collecting)

| Item | Decision |
|---|---|
| **Window** | 1 Jan 2022 to 13 Oct 2026 (the period when the Part 2 gap is largest, plus a pre-surge year of baseline). The analysis cutoff is the same as Part 1. |
| **Search terms** | economy, inflation, prices, jobs, unemployment, recession, cost of living, tariffs, interest rates, gas prices, grocery, mortgage, wages. Written once in the next cell and saved with a timestamp **before** any download. |
| **Unit of observation** | One *item*: a news article (headline + standfirst, and body text where the source gives it) or one Reddit post (title + text). Aggregated to *month* for comparison with the Michigan gap. |
| **Sources** | (1) **The Guardian** (API, full text). A UK-based national outlet with a large US desk. (2) **GDELT** (free news index): US-sourced English articles, **headlines only**, **last ~85 days only** because the free API keeps a rolling 3-month window. (3) **Reddit** via the Arctic Shift public archive: five subreddits spanning the political spectrum, sampled week by week. |
| **Inclusion** | English; economic-topic match in title or standfirst; US-focused (Guardian: US production office or US section; GDELT: US source country; Reddit: US-centred subreddits). |
| **Exclusion / handling** | Exact-URL duplicates removed. Near-duplicate titles (same normalised headline, same source, within 7 days) collapsed to one item. Deleted or removed Reddit posts and `[deleted]` authors flagged and dropped. Items kept only once with the count of copies stored. |
| **What each source can and cannot say** | *Supply* = what outlets published (all three sources). *Engagement* = what people reacted to (Reddit score and comments only). *Exposure* = what a typical household actually saw. **None of these sources measures exposure**, so any later result is about supply and engagement, not about what people saw. |
| **Known gaps (stated up front)** | No historical **US local news** (the free APIs used here do not offer it). GDELT only covers the recent 85 days. Reddit users are not a random sample of voters. The Guardian is not a US outlet. No Fox, CNN, NYT or TikTok. Reddit data is sampled (the archive returns at most 100 to 1,000 posts per weekly request, so very busy subreddits such as r/politics can still be truncated), not complete. |
| **Claims this corpus can support** | Differences in framing between the outlets and communities collected, and how that framing moves with the economy. It cannot support claims about US media as a whole, nor about local news. |

In [ ]:
# 3.0a  Write the protocol to disk BEFORE collecting anything.
SEARCH_TERMS = ["economy", "inflation", "prices", "jobs", "unemployment", "recession", "cost of living",
                "tariffs", "interest rates", "gas prices", "grocery", "mortgage", "wages"]
MEDIA_START = pd.Timestamp("2022-01-01")
MEDIA_END   = CUTOFF                                   # 2026-10-13
GDELT_DAYS  = 85                                       # GDELT keeps ~3 months; stay inside it
SUBREDDITS  = ["Economics", "economy", "personalfinance", "politics", "Conservative"]
REDDIT_CAP  = 100                                      # posts per subreddit per half-month window
GUARDIAN_PAGE_SIZE, GUARDIAN_MAX_PAGES = 50, 6         # up to 300 articles per month, ordered by relevance
GUARDIAN_URL = os.environ.get("GUARDIAN_URL", "https://content.guardianapis.com/search")
GDELT_URL    = os.environ.get("GDELT_URL",    "https://api.gdeltproject.org/api/v2/doc/doc")
ARCTIC_URL   = os.environ.get("ARCTIC_URL",   "https://arctic-shift.photon-reddit.com/api/posts/search")

protocol = {
    "written_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "search_terms": SEARCH_TERMS, "window": [str(MEDIA_START.date()), str(MEDIA_END.date())],
    "guardian": {"sections": "us-news|us-money|business", "page_size": GUARDIAN_PAGE_SIZE,
                 "max_pages_per_month": GUARDIAN_MAX_PAGES, "order": "relevance"},
    "gdelt": {"days": GDELT_DAYS, "window_days": 3, "maxrecords": 250, "filters": "sourcecountry:US sourcelang:english"},
    "reddit": {"subreddits": SUBREDDITS, "window": "weekly", "limit": "auto (100-1000 per request)", "source": "Arctic Shift archive"},
    "dedupe": "exact URL; normalised title within source and 7 days",
    "known_gaps": ["no historical US local news", "GDELT recent 85 days only", "Reddit sampled", "Guardian is UK-based",
                   "no exposure measure"],
}
proto_path = DATA_PROC / "day3_protocol.json"
AMENDMENT_1 = {"id": 1, "note": "Reddit windows changed from half-month (100 posts) to weekly (limit=auto, 100-1000). "
               "Reason: in the first run 100% of half-month windows hit the 100-post cap, so only the first day or so "
               "of each half-month was sampled. Decided after seeing collection counts only, before any text was read or coded."}
if proto_path.exists():                                # never overwrite: the point is that it was fixed in advance
    protocol = json.loads(proto_path.read_text(encoding="utf-8"))
    print("Protocol already on disk (written", protocol["written_utc"] + "). Original not overwritten.")
    if not any(a.get("id") == 1 for a in protocol.get("amendments", [])):
        protocol.setdefault("amendments", []).append({**AMENDMENT_1, "utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")})
        proto_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")
        print("Amendment 1 recorded (Reddit weekly windows).")
else:
    proto_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")
    print("Protocol written", protocol["written_utc"])
print("Terms:", ", ".join(SEARCH_TERMS))

## 3.1 The Guardian (full-text news)
The key is read from the environment variable `GUARDIAN_API_KEY`, or from a line `GUARDIAN_API_KEY=your-key` in a `.env` file in the project folder (git-ignored), or from a one-line `guardian_key.txt`. It is **never** written to the cache manifest.

To set it in VS Code on Windows, open the terminal in the project folder and run `setx GUARDIAN_API_KEY "your-key"`, then **restart VS Code**. The file route needs no restart.

In [ ]:
def get_guardian_key():
    """Order: environment variable, then a line GUARDIAN_API_KEY=... in .env, then guardian_key.txt."""
    k = os.environ.get("GUARDIAN_API_KEY", "").strip()
    envf = ROOT / ".env"
    if not k and envf.exists():
        for line in envf.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if line.startswith("export "): line = line[7:]
            if "=" in line and not line.startswith("#"):
                name, val = line.split("=", 1)
                if name.strip() in ("GUARDIAN_API_KEY", "GUARDIAN_KEY"):
                    k = val.strip().strip('"').strip("'")
    f = ROOT / "guardian_key.txt"
    if not k and f.exists():
        k = f.read_text(encoding="utf-8-sig").strip()
    return k

GUARDIAN_KEY = get_guardian_key()
if GUARDIAN_KEY: _SECRETS.add(GUARDIAN_KEY)
G_QUERY = " OR ".join(f'"{t}"' for t in SEARCH_TERMS)
months = pd.period_range(MEDIA_START, MEDIA_END, freq="M")
guardian_files, guardian_status = [], {}

if not GUARDIAN_KEY:
    print(f"No Guardian key found (looked in the environment, {ROOT / '.env'} and guardian_key.txt). Skipping the Guardian download (cached pages, if any, are still used below).")
    guardian_files = sorted(DATA_CACHE.glob("guardian_*.json"))
else:
    for per in months:
        d0 = max(per.start_time, MEDIA_START).date(); d1 = min(per.end_time, MEDIA_END).date()
        page, pages = 1, 1
        while page <= min(pages, GUARDIAN_MAX_PAGES):
            name = f"guardian_{per}_p{page}"
            path, st = fetch_cached(name, GUARDIAN_URL, suffix=".json", params={
                "q": G_QUERY, "section": "us-news|us-money|business", "from-date": str(d0), "to-date": str(d1),
                "order-by": "relevance", "page-size": GUARDIAN_PAGE_SIZE, "page": page,
                "show-fields": "headline,trailText,bodyText,wordcount,productionOffice",
                "api-key": GUARDIAN_KEY})
            guardian_status[name] = st
            if path is None:
                break
            guardian_files.append(path)
            try:
                pages = json.loads(path.read_text(encoding="utf-8"))["response"].get("pages", 1)
            except Exception:
                pages = 1
            if st == "downloaded":
                time.sleep(0.2)                       # stay well under the API rate limit
            page += 1
        if guardian_status and list(guardian_status.values())[-1].startswith(("failed", "skipped")):
            print("Stopping Guardian collection for this run (see message above). Re-run to resume from the cache.")
            break
    print(f"Guardian pages available: {len(guardian_files)}  | new downloads this run: "
          f"{sum(v == 'downloaded' for v in guardian_status.values())}")

In [ ]:
# Shared by the GDELT and Reddit collectors: wait between requests, back off when the server says "slow down" (429),
# and print the server's own error text once so a problem can be diagnosed instead of guessed at.
_DIAGNOSED = set()
def polite_fetch(name, url, params, gap, tries=4, backoff=20, single_shot=False):
    host = urlparse(url).netloc
    if (DATA_CACHE / f"{name}.json").exists() and not FORCE_REFRESH:
        return fetch_cached(name, url, suffix=".json", params=params)
    st = "failed"
    # single_shot: exactly ONE request per attempt (no extra retries, no curl). Needed for servers with strict limits,
    # where a burst of automatic retries is itself what triggers the 429.
    for k in range(tries):
        _FAILED_HOSTS.discard(host)                       # allow a fresh try instead of fail-fast skipping
        time.sleep(gap if k == 0 else backoff * k)
        path, st = fetch_cached(name, url, suffix=".json", params=params,
                                **({"retries": 1, "curl_fallback": False} if single_shot else {}))
        if path is not None:
            return path, st
    if host not in _DIAGNOSED:
        _DIAGNOSED.add(host)
        try:
            r = requests.get(url, params=params, timeout=30, headers=BROWSER_HEADERS)
            print(f"  server said for {name}: HTTP {r.status_code}: {_scrub(r.text[:300])}")
        except Exception as e:
            print(f"  diagnostic request failed: {_scrub(str(e))[:150]}")
    return None, st

## 3.2 GDELT (recent US headlines, headlines only)
GDELT's free API only looks back about three months, so this source cannot give history. It is used for one thing: a **recent snapshot** that includes many non-national US outlets (regional and local sites are in GDELT). It returns title, URL, domain and time, with no article text.

In [ ]:
gdelt_files = []
g_end = MEDIA_END + pd.Timedelta(days=1)
g_start = MEDIA_END - pd.Timedelta(days=GDELT_DAYS)
gq = "(" + " OR ".join(f'"{t}"' if " " in t else t for t in SEARCH_TERMS) + ") sourcecountry:US sourcelang:english"
w0 = g_start
while w0 < g_end:
    w1 = min(w0 + pd.Timedelta(days=3), g_end)
    name = f"gdelt_{w0:%Y%m%d}_{w1:%Y%m%d}"
    path, st = polite_fetch(name, GDELT_URL, {
        "query": gq, "mode": "ArtList", "format": "json", "maxrecords": 250, "sort": "DateDesc",
        "startdatetime": f"{w0:%Y%m%d}000000", "enddatetime": f"{w1:%Y%m%d}000000"}, gap=7, backoff=30, tries=5, single_shot=True)
    if path is None:
        print("GDELT still refusing after retries; continuing with other sources. Re-run this cell later to resume."); break
    gdelt_files.append(path)
    w0 = w1
print(f"GDELT windows available: {len(gdelt_files)}")

## 3.3 Reddit (social posts, sampled)
Reddit's own API no longer gives free historical access, so this uses the Arctic Shift public archive. **This is the one source I could not test against the live service.** If the cell below reports failures, paste the message and the endpoint can be adjusted. The rest of the notebook still works without it.

In [ ]:
reddit_files = []
# Weekly windows (amendment 1, see protocol file): the first run used half-month windows capped at 100 posts, and every
# window hit the cap, which meant only the first day or so of each half-month was sampled.
halves = [(a, min(a + pd.Timedelta(days=7), MEDIA_END + pd.Timedelta(days=1)))
          for a in pd.date_range(MEDIA_START, MEDIA_END, freq="7D")]
reddit_missing, fails_in_row = [], 0
for sub in SUBREDDITS:
    for a, b in halves:
        if a > MEDIA_END: continue
        name = f"reddit_{sub}_w{a:%Y%m%d}"
        path, st = polite_fetch(name, ARCTIC_URL, {
            "subreddit": sub, "after": f"{a:%Y-%m-%d}", "before": f"{b:%Y-%m-%d}", "limit": "auto", "sort": "asc"},
            gap=0.5, tries=3, backoff=15)
        if path is None:
            reddit_missing.append(name); fails_in_row += 1
            if fails_in_row >= 6:
                print("6 failures in a row: stopping Reddit for this run. Re-run this cell later to resume from the cache."); break
            continue
        fails_in_row = 0
        reddit_files.append(path)
    else:
        continue
    break
if reddit_missing:
    print(f"Reddit windows that could not be fetched: {len(reddit_missing)} (first: {reddit_missing[:3]}). Re-run to retry them.")
print(f"Reddit windows available: {len(reddit_files)}")

## 3.4 Normalise, filter, dedupe
Every source is turned into the same columns, then passed through the protocol's rules. The **funnel table** shows how many items each rule removed, so nothing disappears silently.

`scope` is a coarse label used later to ask *whose* coverage a result describes: `guardian_national` (UK-based national outlet), `us_domain_national` / `us_domain_other` (GDELT: a short hand-made list of national US outlets vs. every other US domain, which includes local and regional sites), and `social`.

In [ ]:
NATIONAL_US = {"nytimes.com", "washingtonpost.com", "wsj.com", "cnn.com", "foxnews.com", "nbcnews.com", "cbsnews.com",
               "abcnews.go.com", "npr.org", "usatoday.com", "politico.com", "apnews.com", "reuters.com", "bloomberg.com",
               "cnbc.com", "msnbc.com", "thehill.com", "axios.com", "forbes.com", "businessinsider.com", "newsweek.com",
               "huffpost.com", "breitbart.com", "dailywire.com", "nypost.com", "yahoo.com", "marketwatch.com"}
TERM_RE = re.compile("|".join(re.escape(t) for t in SEARCH_TERMS) + r"|price|job|wage|econom|inflat|tariff|rates?\b", re.I)

def clean(s):
    s = re.sub(r"<[^>]+>", " ", str(s or ""))
    return re.sub(r"\s+", " ", s).strip()

rows = []
# --- Guardian
for f in guardian_files:
    try: res = json.loads(f.read_text(encoding="utf-8"))["response"]["results"]
    except Exception: continue
    for r in res:
        fl = r.get("fields", {}) or {}
        body = clean(fl.get("bodyText"))
        rows.append(dict(platform="guardian", outlet="theguardian.com", scope="guardian_national",
                         date=pd.to_datetime(r.get("webPublicationDate"), utc=True, errors="coerce"),
                         title=clean(fl.get("headline") or r.get("webTitle")), snippet=clean(fl.get("trailText")),
                         text=body[:2000], body_chars=len(body), url=r.get("webUrl"), section=r.get("sectionId"),
                         us_focus=(fl.get("productionOffice") == "US") or r.get("sectionId") in ("us-news", "us-money"),
                         engagement=np.nan, author_hash=None, text_level="full_article" if body else "headline_only"))
# --- GDELT
for f in gdelt_files:
    try: arts = json.loads(f.read_text(encoding="utf-8")).get("articles", [])
    except Exception: continue
    for r in arts:
        dom = (r.get("domain") or "").lower().removeprefix("www.")
        rows.append(dict(platform="gdelt", outlet=dom, scope="us_domain_national" if dom in NATIONAL_US else "us_domain_other",
                         date=pd.to_datetime(r.get("seendate"), format="%Y%m%dT%H%M%SZ", utc=True, errors="coerce"),
                         title=clean(r.get("title")), snippet="", text="", body_chars=0, url=r.get("url"), section=None,
                         us_focus=(r.get("sourcecountry", "United States") in ("United States", "US")),
                         engagement=np.nan, author_hash=None, text_level="headline_only"))
# --- Reddit
for f in reddit_files:
    try: posts = json.loads(f.read_text(encoding="utf-8")).get("data", [])
    except Exception: continue
    sub_name = f.name.split("_")[1]
    for r in posts:
        selftext = clean(r.get("selftext"))
        removed = selftext in ("[removed]", "[deleted]") or r.get("author") in (None, "[deleted]")
        a = str(r.get("author") or "")
        rows.append(dict(platform="reddit", outlet="r/" + sub_name, scope="social",
                         date=pd.to_datetime(r.get("created_utc"), unit="s", utc=True, errors="coerce"),
                         title=clean(r.get("title")), snippet="", text="" if removed else selftext[:2000],
                         body_chars=0 if removed else len(selftext), url=r.get("permalink") or r.get("url"), section=None,
                         us_focus=True, engagement=(r.get("score") or 0) + (r.get("num_comments") or 0),
                         author_hash=hashlib.sha1(a.encode()).hexdigest()[:10] if a not in ("", "[deleted]") else None,
                         text_level="post", removed_or_deleted=removed))
raw = pd.DataFrame(rows)
assert not raw.empty, "No media items collected yet. Run the collection cells above first and read their messages."
if "removed_or_deleted" not in raw: raw["removed_or_deleted"] = False
raw["removed_or_deleted"] = raw["removed_or_deleted"].fillna(False).astype(bool)

funnel = [("collected", len(raw))]
c = raw.dropna(subset=["date", "title"]);                                         funnel.append(("has a date and title", len(c)))
c = c[(c.date >= MEDIA_START.tz_localize("UTC")) & (c.date < (MEDIA_END + pd.Timedelta(days=1)).tz_localize("UTC"))]
funnel.append(("inside the date window", len(c)))
c = c[c.title.str.len() > 0];                                                      funnel.append(("non-empty title", len(c)))
c = c[c.us_focus];                                                                 funnel.append(("US-focused", len(c)))
c = c[c.title.str.contains(TERM_RE) | c.snippet.str.contains(TERM_RE)];            funnel.append(("economic term in title or standfirst", len(c)))
c = c[~c.removed_or_deleted];                                                      funnel.append(("not deleted/removed (Reddit)", len(c)))
c = c.drop_duplicates(subset=["url"]);                                             funnel.append(("unique URL", len(c)))
c = c.assign(title_key=c.title.str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip(),
             week=c.date.dt.tz_convert(None).dt.to_period("W"))
c["copies"] = c.groupby(["platform", "outlet", "title_key", "week"]).url.transform("size")
c = c.drop_duplicates(subset=["platform", "outlet", "title_key", "week"]);         funnel.append(("near-duplicate titles collapsed", len(c)))
corpus = c.drop(columns=["title_key", "week"]).sort_values("date").reset_index(drop=True)
corpus.insert(0, "item_id", [f"{p[0].upper()}{i:06d}" for i, p in enumerate(corpus.platform)])
corpus["month"] = corpus.date.dt.tz_convert(None).dt.to_period("M").dt.to_timestamp("M")

fun = pd.DataFrame(funnel, columns=["step", "items remaining"])
fun["removed at this step"] = (-fun["items remaining"].diff()).fillna(0).astype(int)
display(fun)
print(f"\nFinal corpus: {len(corpus):,} items")
display(corpus.groupby(["platform", "scope"]).size().rename("items").to_frame())

## 3.5 Corpus quality checks
What matters here is not volume but whether the corpus is lopsided in ways that would distort Day 4–5. Each check below says what it is looking for.

In [ ]:
INK, GREY, BLUE, ORANGE, AQUA = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a"
# (a) items per month per platform: gaps and sudden jumps would be collection artefacts, not news
mc = corpus.groupby(["month", "platform"]).size().unstack(fill_value=0).reindex(
    pd.date_range(MEDIA_START, MEDIA_END, freq="ME"), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
for col, colr in zip(mc.columns, [BLUE, ORANGE, AQUA]):
    ax.plot(mc.index, mc[col], color=colr, lw=1.6, label=col)
ax.set_title("Items kept per month, by platform", loc="left", color=INK)
ax.set_ylabel("items"); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(DATA_PROC / "day3_items_per_month.png", dpi=130); plt.show()

# (b) months with no or very few items
thin = mc[(mc.sum(axis=1) < 20)]
print(f"Months with fewer than 20 items in total: {len(thin)} of {len(mc)}")
if len(thin): print("  ", ", ".join(f"{d:%Y-%m}" for d in thin.index[:12]), "..." if len(thin) > 12 else "")

# (c) how much of each source is full text vs headline only
print("\nText available:")
display(corpus.groupby("platform").agg(items=("item_id", "size"), share_with_body=("body_chars", lambda s: (s > 0).mean()),
                                       median_title_chars=("title", lambda s: s.str.len().median()),
                                       median_body_chars=("body_chars", "median")).round(2))

# (d) concentration: does one outlet or one user dominate?
print("\nTop outlets (share of all items):")
display((corpus.outlet.value_counts(normalize=True).head(8) * 100).round(1).rename("% of items").to_frame())
rd = corpus[corpus.platform == "reddit"]
if len(rd):
    top10 = rd.author_hash.value_counts(normalize=True).head(10).sum()
    print(f"Reddit: the 10 most active authors account for {top10:.0%} of kept posts "
          f"({'high: check for automated posters before Day 4' if top10 > 0.25 else 'no heavy concentration'}).")
    print(f"Reddit: {(rd.copies > 1).mean():.0%} of posts had repeated titles in the same week.")

# (e) political and geographic composition of the sources
print("\nCoverage by scope:")
display(corpus.scope.value_counts().rename("items").to_frame())
if (corpus.scope == "us_domain_other").sum() == 0:
    print("NOTE: no non-national US domains (local/regional) are in the corpus this run. Local news is NOT represented.")

# (e2) Reddit balance: every protocol subreddit should be present, or the social sample is one community's voice
if reddit_files:
    n_win = len([1 for a, _ in halves if a <= MEDIA_END])
    got = pd.Series([f.name.split("_")[1] for f in reddit_files]).value_counts().reindex(SUBREDDITS, fill_value=0)
    cap_hit = pd.Series({s: np.mean([len(json.loads(f.read_text(encoding="utf-8")).get("data", [])) >= REDDIT_CAP
                                     for f in reddit_files if f.name.split("_")[1] == s] or [np.nan]) for s in SUBREDDITS})
    rb = pd.DataFrame({"windows fetched": got, "of": n_win, "share of weeks with 100+ posts (may be truncated)": cap_hit.round(2)})
    print("Reddit windows by subreddit:"); display(rb)
    if (got < 0.9 * n_win).any():
        print("WARNING: some subreddits are missing or incomplete. Re-run the Reddit cell until all five are complete "
              "before using Reddit in any comparison; right now the social sample is not balanced across communities.")
    if (cap_hit > 0.5).any():
        print("NOTE: in weeks where the archive returned its maximum, the sample is the EARLIEST posts of that week, not all posts. Treat busy subreddits (r/politics, r/economy) as truncated.")

# (f) overlap with the Michigan panel months
print(f"\nMichigan months covered by at least one item: "
      f"{int((mc.sum(axis=1) > 0).sum())} of {len(mc)} between {MEDIA_START:%b %Y} and {MEDIA_END:%b %Y}")

In [ ]:
# 3.6  Save corpus + a summary the report can quote
out_cols = ["item_id", "platform", "outlet", "scope", "date", "month", "title", "snippet", "text", "text_level",
            "body_chars", "url", "section", "engagement", "copies"]
corpus[out_cols].to_csv(DATA_PROC / "day3_corpus.csv", index=False, encoding="utf-8")
fun.to_csv(DATA_PROC / "day3_funnel.csv", index=False, encoding="utf-8")
mc.to_csv(DATA_PROC / "day3_items_per_month.csv", encoding="utf-8")
print("Saved:", ", ".join(p.name for p in sorted(DATA_PROC.glob("day3_*"))))
print("\nDAY 3 SUMMARY")
for p_, n_ in corpus.platform.value_counts().items():
    print(f"- {p_}: {n_:,} items kept ({corpus[corpus.platform == p_].date.min():%b %Y} to {corpus[corpus.platform == p_].date.max():%b %Y}).")
print("- This corpus measures what was PUBLISHED or POSTED (supply, and Reddit engagement). It does not measure what households saw.")
print("- Not covered: historical US local news, TikTok/X/Facebook, US national TV. Any result must be worded for the sources actually collected.")

## Open items carried into Day 4
* Write the **coding scheme** (tone, what is emphasised, framing, factual accuracy against the data, political attribution) *before* any model or person labels anything.
* Draw a **stratified sample of at least 300 items** from `day3_corpus.csv` (balanced across platform, scope and period).
* Decide on a second rater. With one human rater only, agreement can only be estimated by labelling the same sample twice at least 3 days apart, plus agreement with an AI rater. This is a weaker check and will be reported as a limitation.
* Build the dictionary (word-list) baseline so the AI method has something to beat.

---
# Part 4 (Day 4): Coding scheme and labeling sample

**Order of work (fixed in advance, so the AI method cannot be tuned to the human labels):**
1. The labeling guide (`LABELING_GUIDE.md`) is written and frozen. Version 1, committed to the repo.
2. A stratified sample of 300 items is drawn with a fixed random seed.
3. The human labels all 300 from a blind sheet (outlet and date hidden).
4. Only then are the AI labels and the word-list baseline produced, blind to the human labels.
5. Agreement is measured (Cohen's kappa; quadratic-weighted kappa for the ordinal `tone`). Where a second human rater is not available, a second pass by the same person at least 3 days later plus AI agreement are used, and this is disclosed as a weaker check.

**The five things coded** (details in the guide): US-economy relevance, topic, tone (-2 to +2), dominant frame, who is blamed or credited.

**Why the sample is not proportional.** Reddit is 90% of the corpus and the political subreddits are small, so a proportional sample would hold almost no Guardian or political-subreddit items. Quotas are set per source and spread evenly over three periods. Each item keeps a weight (`stratum_size / n_sampled`) so any population-level estimate can be re-weighted.

In [ ]:
# 4.1  Draw the labeling sample (fixed seed, deterministic)
SEED = 20261015
corpus_df = pd.read_csv(DATA_PROC / "day3_corpus.csv", parse_dates=["date"])
pool = corpus_df[corpus_df.platform.isin(["guardian", "reddit"]) & (corpus_df.title.str.len() >= 25)].copy()
pool["period"] = pd.cut(pool.date.dt.year, [2021, 2023, 2025, 2026], labels=["2022-23", "2024-25", "2026"])
QUOTA = {"theguardian.com": 120, "r/Economics": 55, "r/economy": 45, "r/personalfinance": 30, "r/politics": 25, "r/Conservative": 25}
assert sum(QUOTA.values()) == 300
parts = []
for outlet, n in QUOTA.items():
    g = pool[pool.outlet == outlet]
    per_period = n // 3
    for k, (per, gg) in enumerate(g.groupby("period", observed=True)):
        take = per_period + (n - 3 * per_period if k == 0 else 0)          # remainder goes to the first period
        s = gg.sample(n=min(take, len(gg)), random_state=SEED)
        s = s.assign(stratum=f"{outlet}|{per}", stratum_size=len(gg), n_sampled=len(s))
        parts.append(s)
sample = pd.concat(parts)
sample["weight"] = sample.stratum_size / sample.n_sampled
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)       # random order
sample.insert(0, "label_id", [f"L{i + 1:03d}" for i in range(len(sample))])
rng = np.random.default_rng(SEED)
sample["second_pass"] = 0
sample.loc[rng.choice(len(sample), size=100, replace=False), "second_pass"] = 1
print(f"Sample: {len(sample)} items; second-pass subset: {int(sample.second_pass.sum())}")
display(sample.groupby(["outlet", "period"], observed=True).size().unstack(fill_value=0))
short = {o: int((sample.outlet == o).sum()) for o in QUOTA if (sample.outlet == o).sum() != QUOTA[o]}
print("All quotas met." if not short else f"Quotas NOT met (outlet: n): {short}")

In [ ]:
# 4.2  Write the blind label sheet (xlsx with dropdowns) and the key (kept separate on purpose)
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment, Font

def excerpt(r):
    t = str(r.text) if isinstance(r.text, str) else ""
    return t[:600]
sheet = pd.DataFrame({
    "label_id": sample.label_id, "title": sample.title, "standfirst": sample.snippet.fillna(""),
    "text_excerpt": sample.apply(excerpt, axis=1),
    "us_econ": "", "topic": "", "tone": "", "frame": "", "blame": "", "confidence": "", "note": ""})
OPTIONS = {"us_econ": "1,0", "topic": "prices_inflation,jobs_wages,tariffs_trade,rates_housing,growth_recession,other_econ",
           "tone": "-2,-1,0,1,2", "frame": "decline_crisis,cost_burden,resilience_recovery,risk_uncertainty,political_blame,neutral_report",
           "blame": "none,trump_gop,biden_dems,fed,business,foreign,other", "confidence": "1,2,3"}
def write_sheet(df, path):
    wb = Workbook(); ws = wb.active; ws.title = "labels"
    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append(["" if (isinstance(v, float) and np.isnan(v)) else v for v in row])
    for col, opts in OPTIONS.items():
        letter = chr(ord("A") + list(df.columns).index(col))
        dv = DataValidation(type="list", formula1=f'"{opts}"', allow_blank=True, showErrorMessage=True)
        ws.add_data_validation(dv); dv.add(f"{letter}2:{letter}{len(df) + 1}")
    for letter, w in zip("ABCDEFGHIJK", [9, 55, 45, 60, 8, 18, 6, 20, 12, 10, 25]):
        ws.column_dimensions[letter].width = w
    for r in ws.iter_rows(min_row=2):
        for c in r[1:4]: c.alignment = Alignment(wrap_text=True, vertical="top")
    for c in ws[1]: c.font = Font(bold=True)
    ws.freeze_panes = "B2"
    wb.save(path)

write_sheet(sheet, DATA_PROC / "day4_label_sheet_pass1.xlsx")
write_sheet(sheet[sample.second_pass.eq(1).to_numpy()].reset_index(drop=True), DATA_PROC / "day4_label_sheet_pass2_BLANK.xlsx")
key = sample[["label_id", "item_id", "platform", "outlet", "date", "period", "stratum", "stratum_size", "n_sampled", "weight", "second_pass", "url"]]
key.to_csv(DATA_PROC / "day4_label_key.csv", index=False, encoding="utf-8")
print("Saved:", ", ".join(p.name for p in sorted(DATA_PROC.glob("day4_*"))))
print("Pass 1 sheet (300 items) is for you to label now. Pass 2 is a blank copy of 100 items: use it at least 3 days later.")

## 4.3 Load and check the human labels
Expected file: `data/processed/day4_labels_HUMAN_pass1_filled.xlsx`. Every row must be complete and use only allowed values. Rows marked `us_econ = 0` must have the later columns empty.

In [ ]:
HUMAN_FILE = DATA_PROC / "day4_labels_HUMAN_pass1_filled.xlsx"
ALLOWED = {"topic": ["prices_inflation", "jobs_wages", "tariffs_trade", "rates_housing", "growth_recession", "other_econ"],
           "tone": ["-2", "-1", "0", "1", "2"],
           "frame": ["decline_crisis", "cost_burden", "resilience_recovery", "risk_uncertainty", "political_blame", "neutral_report"],
           "blame": ["none", "trump_gop", "biden_dems", "fed", "business", "foreign", "other"], "confidence": ["1", "2", "3"]}
hum = pd.read_excel(HUMAN_FILE, dtype=str).fillna("")
hum = hum.apply(lambda s: s.str.strip())
for c in ("tone", "us_econ", "confidence"):
    hum[c] = hum[c].str.replace(r"\.0$", "", regex=True)
assert len(hum) == 300 and hum.label_id.is_unique, "expected 300 unique rows"
assert hum.us_econ.isin(["0", "1"]).all(), "us_econ must be 0 or 1 in every row"
us = hum.us_econ == "1"
for c, ok in ALLOWED.items():
    bad = hum[us & ~hum[c].isin(ok)]
    assert bad.empty, f"{c}: invalid or blank values in rows {bad.label_id.tolist()[:10]}"
assert (hum[~us][list(ALLOWED)] == "").all().all(), "rows with us_econ=0 should leave later columns empty"
print(f"Human labels OK: {len(hum)} rows, {int(us.sum())} about the US economy, {int((~us).sum())} not.")
for c in ["tone", "frame", "blame", "topic"]:
    print(f"\n{c}:", hum.loc[us, c].value_counts().to_dict())
hum.to_csv(DATA_PROC / "day4_human_labels_clean.csv", index=False, encoding="utf-8")

## 4.4 AI labels (OpenAI API)
The same guide, the same blind view (headline, standfirst, text excerpt; no outlet, no date). **Settings are fixed here, before any result is seen:** temperature 0, JSON output restricted to the allowed values, one call per item, the exact prompt saved and hashed, the exact model snapshot the API reports saved with every label.

**Setup:** put `OPENAI_API_KEY=sk-...` in the `.env` file next to the notebook. The key is never written to the repo or the manifest. Every answer is cached in a `.jsonl` file, so a re-run costs nothing, and a **spending cap** stops the run before it overspends. Check the real bill at platform.openai.com/usage.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

def get_secret(name):
    v = os.environ.get(name, "").strip()
    envf = ROOT / ".env"
    if not v and envf.exists():
        for line in envf.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip().removeprefix("export ")
            if "=" in line and not line.startswith("#"):
                k, val = line.split("=", 1)
                if k.strip() == name: v = val.strip().strip('"').strip("'")
    return v

OPENAI_KEY = get_secret("OPENAI_API_KEY")
if OPENAI_KEY: _SECRETS.add(OPENAI_KEY)
OPENAI_URL = os.environ.get("OPENAI_URL", "https://api.openai.com/v1/chat/completions")
MODEL = "gpt-4.1-mini"
PRICE_IN, PRICE_OUT = 0.40 / 1e6, 1.60 / 1e6        # USD per token for gpt-4.1-mini (third-party price page, 15 Oct 2026; verify on openai.com)
BUDGET_SAMPLE_USD = 0.60                            # hard cap for labeling the 300-item sample (expected cost about $0.15)

SYSTEM_PROMPT = """You are a careful annotator. You will see ONE news headline or social-media post about the economy (headline, standfirst, text excerpt). Label it using ONLY that text. Do not use outside knowledge about the source. Label what the text says, not whether it is true.

Fields:
us_econ: 1 if it is about US economic conditions or policy that affects Americans' finances (prices, jobs, wages, rates, housing, tariffs, growth, the Fed). 0 if it is about a non-US economy, markets-only trivia, or not about economic conditions (pure ideology, jokes, personal stories with no economic claim). If 0, set topic, tone, frame, blame to null.
topic: prices_inflation | jobs_wages | tariffs_trade | rates_housing | growth_recession | other_econ (main subject).
tone: integer -2..2, how the item portrays economic conditions for ordinary Americans. -2 very negative (crisis, collapse, hardship); -1 negative or worrying; 0 neutral, mixed or no judgement; 1 positive; 2 very positive. Judge the portrayal, not the topic ("inflation falls to 2%" is positive). A question headline takes the implied stance, or 0 if none.
frame: the dominant way the story is told: decline_crisis (things getting worse overall) | cost_burden (affordability and living costs hurting households) | resilience_recovery (economy holding up or improving) | risk_uncertainty (something bad might happen) | political_blame (main point is who is responsible) | neutral_report (states data or events without a story).
blame: who the text itself holds responsible (or credits) for the economic situation: none | trump_gop | biden_dems | fed | business | foreign | other. Use none unless the text assigns responsibility.
confidence: 1 guess, 2 fairly sure, 3 sure."""
PROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:8]
(DATA_PROC / "day4_ai_prompt.txt").write_text(f"MODEL: {MODEL}\nTEMPERATURE: 0\nPROMPT_HASH: {PROMPT_HASH}\n\n{SYSTEM_PROMPT}\n", encoding="utf-8")

def _enum(vals): return {"type": ["string", "null"], "enum": vals + [None]}
SCHEMA = {"type": "object", "additionalProperties": False,
          "required": ["us_econ", "topic", "tone", "frame", "blame", "confidence"],
          "properties": {"us_econ": {"type": "integer", "enum": [0, 1]},
                         "topic": _enum(ALLOWED["topic"]), "frame": _enum(ALLOWED["frame"]), "blame": _enum(ALLOWED["blame"]),
                         "tone": {"type": ["integer", "null"], "enum": [-2, -1, 0, 1, 2, None]},
                         "confidence": {"type": "integer", "enum": [1, 2, 3]}}}

def item_text(r):
    f = lambda v: "-" if (v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == "") else str(v)
    return f"HEADLINE: {f(r['title'])}\nSTANDFIRST: {f(r['standfirst'])}\nEXCERPT: {f(r['text_excerpt'])}"

_lock = threading.Lock()
def _call_api(text):
    body = {"model": MODEL, "temperature": 0, "seed": 7,
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}],
            "response_format": {"type": "json_schema", "json_schema": {"name": "label", "strict": True, "schema": SCHEMA}}}
    for attempt in range(6):
        try:
            r = requests.post(OPENAI_URL, json=body, timeout=60, headers={"Authorization": f"Bearer {OPENAI_KEY}"})
        except Exception as e:
            time.sleep(2 ** attempt); last = _scrub(str(e))[:150]; continue
        if r.status_code == 200:
            return r.json()
        if r.status_code == 400 and "temperature" in r.text and "temperature" in body:
            body.pop("temperature"); continue                     # some newer models only accept the default
        if r.status_code in (429, 500, 502, 503):
            time.sleep(float(r.headers.get("retry-after", 2 ** attempt))); last = f"HTTP {r.status_code}"; continue
        raise RuntimeError(f"OpenAI error {r.status_code}: {_scrub(r.text[:300])}")
    raise RuntimeError(f"OpenAI request failed after retries: {last}")

def ai_label(items, tag, budget_usd):
    """items: DataFrame with columns key, title, standfirst, text_excerpt. Cached in a .jsonl; spends at most budget_usd."""
    path = DATA_CACHE / f"ai_{MODEL}_{PROMPT_HASH}_{tag}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip(): d = json.loads(line); done[d["key"]] = d
    spent = sum(d["usage"]["prompt_tokens"] * PRICE_IN + d["usage"]["completion_tokens"] * PRICE_OUT for d in done.values())
    todo = [r for _, r in items.iterrows() if r["key"] not in done]
    print(f"{tag}: {len(done)} cached, {len(todo)} to label; spent so far (estimate) ${spent:.3f}, cap ${budget_usd:.2f}")
    if todo and not OPENAI_KEY:
        print("  No OPENAI_API_KEY found (environment or .env). Skipping new labels."); todo = []
    stop = {"flag": False}; spent_box = [spent]
    def work(r):
        if stop["flag"]: return
        resp = _call_api(item_text(r))
        lab = json.loads(resp["choices"][0]["message"]["content"])
        rec = {"key": r["key"], "label": lab, "usage": resp["usage"], "model_returned": resp.get("model"),
               "utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
        with _lock:
            with path.open("a", encoding="utf-8") as fh: fh.write(json.dumps(rec) + "\n")
            done[r["key"]] = rec
            spent_box[0] += resp["usage"]["prompt_tokens"] * PRICE_IN + resp["usage"]["completion_tokens"] * PRICE_OUT
            if spent_box[0] >= budget_usd: stop["flag"] = True
    if todo:
        with ThreadPoolExecutor(6) as ex: list(ex.map(work, todo))
        print(f"  finished: {len(done)} labeled, estimated spend ${spent_box[0]:.3f}" + ("  (STOPPED AT CAP)" if stop["flag"] else ""))
    out = pd.DataFrame([{"key": k, **{f"ai_{c}": v for c, v in d["label"].items()}, "ai_model": d["model_returned"],
                         "ai_prompt_hash": PROMPT_HASH} for k, d in done.items()])
    return out

sheet_in = pd.read_excel(DATA_PROC / "day4_label_sheet_pass1.xlsx", dtype=str).fillna("")
sheet_in["key"] = sheet_in.label_id
ai_sample = ai_label(sheet_in, "sample300", BUDGET_SAMPLE_USD)
EXT_AI = DATA_PROC / "day4_ai_labels_EXTERNAL_sample.csv"       # optional: labels made outside this notebook (e.g. by Claude in a chat)
if len(ai_sample) == 0 and EXT_AI.exists():
    ai_sample = pd.read_csv(EXT_AI, dtype=str).fillna("").rename(columns={"label_id": "key"})
    assert len(ai_sample) == 300 and ai_sample.key.is_unique, "external AI file must have 300 unique label_id rows"
    print(f"Loaded EXTERNAL AI labels from {EXT_AI.name}: model = {ai_sample.ai_model.unique().tolist()}. "
          "Note: these cannot be re-run by a grader; the prompt and model are recorded in the file.")
elif len(ai_sample):
    ai_sample.rename(columns={"key": "label_id"}).to_csv(DATA_PROC / "day4_ai_labels_sample.csv", index=False, encoding="utf-8")
    print("Models reported by the API:", ai_sample.ai_model.unique().tolist())

## 4.5 Agreement: human vs AI
* **Cohen's kappa** corrects raw agreement for chance. Rough reading: below 0.4 weak, 0.4 to 0.6 moderate, 0.6 to 0.8 substantial, above 0.8 very high.
* `tone` is ordinal, so **quadratic-weighted kappa** is used (a miss by 2 points counts more than a miss by 1).
* Topic, frame and blame are compared only on items both raters call US-economy.
* **95% bootstrap intervals** (2,000 resamples) show how uncertain kappa is with only about 150 items.
* **Bias check:** the average AI tone minus the average human tone. A kappa can be high while one rater is systematically more negative, and that offset matters for a bias study.

In [ ]:
def kappa(a, b, labels, weights=None):
    k = len(labels); idx = {l: i for i, l in enumerate(labels)}
    O = np.zeros((k, k))
    for x, y in zip(a, b): O[idx[x], idx[y]] += 1
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    W = ((np.arange(k)[:, None] - np.arange(k)[None, :]) / (k - 1)) ** 2 if weights == "quadratic" else 1 - np.eye(k)
    return 1 - (W * O).sum() / (W * E).sum() if (W * E).sum() > 0 else np.nan

def kappa_ci(a, b, labels, weights=None, n=2000, seed=1):
    a, b = np.asarray(a), np.asarray(b); rng = np.random.default_rng(seed); ks = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a)); ks.append(kappa(a[i], b[i], labels, weights))
    return np.nanpercentile(ks, [2.5, 97.5])

def agreement_table(h, ai, hcol_prefix="", acol_prefix="ai_"):
    m = h.merge(ai, on="label_id", suffixes=("", "_ai"))
    rows = []
    m["us_econ"] = m.us_econ.astype(int); m["ai_us_econ"] = m.ai_us_econ.astype(int)
    both = m[(m.us_econ == 1) & (m.ai_us_econ == 1)].copy()
    specs = [("us_econ (all items)", m.us_econ.astype(str), m.ai_us_econ.astype(str), ["0", "1"], None),
             ("topic", both.topic, both.ai_topic, ALLOWED["topic"], None),
             ("tone (weighted)", both.tone.astype(int).astype(str), both.ai_tone.astype(int).astype(str), ["-2", "-1", "0", "1", "2"], "quadratic"),
             ("frame", both.frame, both.ai_frame, ALLOWED["frame"], None),
             ("blame", both.blame, both.ai_blame, ALLOWED["blame"], None)]
    for name, a, b, labs, w in specs:
        a, b = a.tolist(), b.tolist(); lo, hi = kappa_ci(a, b, labs, w)
        rows.append({"field": name, "n": len(a), "raw agreement": np.mean(np.array(a) == np.array(b)),
                     "kappa": kappa(a, b, labs, w), "95% CI low": lo, "95% CI high": hi})
    return pd.DataFrame(rows).round(2), both, m

if len(ai_sample):
    agree, both, merged = agreement_table(hum, ai_sample.rename(columns={"key": "label_id"}))
    display(agree)
    off = (both.ai_tone.astype(float) - both.tone.astype(float))
    print(f"Mean tone, human {both.tone.astype(float).mean():+.2f}, AI {both.ai_tone.astype(float).mean():+.2f}; "
          f"AI minus human {off.mean():+.2f} (95% CI {off.mean() - 1.96 * off.std() / np.sqrt(len(off)):+.2f} to {off.mean() + 1.96 * off.std() / np.sqrt(len(off)):+.2f}).")
    print("\nTone: rows = human, columns = AI")
    display(pd.crosstab(both.tone.astype(int), both.ai_tone.astype(int)))
    print("Agreement on tone by how sure the human was:")
    display(both.assign(match=(both.tone.astype(int) == both.ai_tone.astype(int))).groupby("confidence").match.agg(["mean", "size"]).round(2))
    agree.to_csv(DATA_PROC / "day4_agreement_human_vs_ai.csv", index=False)
else:
    print("No AI labels yet: add OPENAI_API_KEY to .env and re-run the cell above.")

### 4.5b Human vs human (second rater, 100 items)
A classmate labeled the 100-item subset from a blank sheet, working alone. File expected at `data/processed/day4_labels_HUMAN2_pass2_filled.xlsx`. This is the ceiling to compare the AI against: if two humans agree at kappa 0.9, an AI at 0.6 is clearly worse than a person; if two humans only reach 0.6, an AI at 0.6 is as good as a person.

In [ ]:
H2_FILE = DATA_PROC / "day4_labels_HUMAN2_pass2_filled.xlsx"
if H2_FILE.exists():
    h2 = pd.read_excel(H2_FILE, dtype=str).fillna("").apply(lambda s: s.str.strip())
    for c in ("tone", "us_econ", "confidence"): h2[c] = h2[c].str.replace(r"\.0$", "", regex=True)
    assert len(h2) == 100 and h2.label_id.is_unique and set(h2.label_id) <= set(hum.label_id), "second-rater file does not match the 100-item subset"
    us2 = h2.us_econ == "1"
    assert h2.us_econ.isin(["0", "1"]).all()
    for c, ok in ALLOWED.items():
        assert h2[us2][c].isin(ok).all(), f"second rater: invalid or blank {c}"
    cols = ["us_econ", "topic", "tone", "frame", "blame", "confidence"]
    h2_as_ai = h2.rename(columns={c: "ai_" + c for c in cols})[["label_id"] + ["ai_" + c for c in cols]]
    hh, both_hh, _ = agreement_table(hum, h2_as_ai)
    display(hh)
    t1, t2 = both_hh.tone.astype(int), both_hh.ai_tone.astype(int)
    print(f"Tone: second rater minus first rater {np.mean(t2 - t1):+.2f}; within one point on {np.mean(abs(t1 - t2) <= 1):.0%} of items.")
    print("Tone: rows = rater 1, columns = rater 2"); display(pd.crosstab(t1, t2))
    hh.to_csv(DATA_PROC / "day4_agreement_human_vs_human.csv", index=False)
else:
    print("Second-rater file not found yet.")

## 4.6 Word-list baseline
A deliberately simple, transparent method that the AI has to beat to be worth its cost. The word lists below are fixed here and were **not** tuned on the human labels. Known weaknesses: no negation, no sarcasm, no context ("prices fall" is positive but "fall" looks negative). A method that cannot beat this is not adding value.

In [ ]:
NEG = ["inflation", "recession", "crisis", "layoffs", "layoff", "unemployment", "soar", "soaring", "surge", "plunge", "slump", "shortage",
       "shortages", "hike", "hikes", "tariffs", "tariff", "debt", "deficit", "fear", "fears", "worry", "worries", "pain", "struggle",
       "struggling", "hurt", "hurts", "cost", "costs", "costly", "expensive", "unaffordable", "cuts", "collapse", "downturn", "risk", "risks",
       "stagflation", "bankruptcy", "poverty", "evict", "eviction", "crash", "warn", "warns", "gloom", "slowdown", "default"]
POS = ["growth", "grows", "recovery", "rebound", "gain", "gains", "boom", "strong", "stronger", "robust", "resilient", "resilience", "improve",
       "improves", "improved", "cooling", "eases", "ease", "easing", "beat", "beats", "record high", "surge in jobs", "hiring", "raises", "raise",
       "wage gains", "optimism", "optimistic", "upbeat", "relief", "soft landing", "falls to", "drop in inflation"]
def dict_score(title, standfirst=""):
    t = f" {title} {standfirst} ".lower()
    t = re.sub(r"[^a-z0-9' ]", " ", t)
    n = sum(len(re.findall(rf"\b{re.escape(w)}\b", t)) for w in NEG)
    p = sum(len(re.findall(rf"\b{re.escape(w)}\b", t)) for w in POS)
    return p - n
hum["dict_score"] = [dict_score(a, b) for a, b in zip(hum.title, hum.standfirst)]
hum["dict_tone3"] = np.sign(hum.dict_score).astype(int)
usd = hum[hum.us_econ == "1"].copy()
usd["h3"] = np.sign(usd.tone.astype(int))
labs3 = [-1, 0, 1]
k_dict = kappa(usd.h3.tolist(), usd.dict_tone3.tolist(), labs3, "quadratic")
rows = [("Word list vs human", k_dict, np.mean(usd.h3 == usd.dict_tone3))]
if len(ai_sample):
    j = usd.merge(ai_sample.rename(columns={"key": "label_id"}), on="label_id")
    j = j[j.ai_us_econ.astype(int) == 1]
    j["a3"] = np.sign(j.ai_tone.astype(float)).astype(int)
    rows += [("AI vs human", kappa(j.h3.tolist(), j.a3.tolist(), labs3, "quadratic"), np.mean(j.h3 == j.a3)),
             ("Word list vs AI", kappa(j.a3.tolist(), j.dict_tone3.tolist(), labs3, "quadratic"), np.mean(j.a3 == j.dict_tone3))]
base = pd.DataFrame(rows, columns=["comparison (3-class tone: negative / neutral / positive)", "weighted kappa", "raw agreement"]).round(2)
display(base)
print("Share of US-economy items where the word list found no sentiment words (scored 0):", f"{(usd.dict_score == 0).mean():.0%}")
base.to_csv(DATA_PROC / "day4_baseline_comparison.csv", index=False)

## 4.7 Swap tests (does the AI react to names rather than content?)
Items that name Trump, Biden or a party are labeled with the political names swapped (Trump <-> Biden, Republican <-> Democrat, GOP <-> Democrats). If the AI is judging content, then:
* **tone should barely move** (a story about prices is as negative with either name);
* **blame should flip sides** (`trump_gop` becomes `biden_dems`) and nothing else should change.

Large tone shifts after a swap would mean the labeler is reacting to who is named, which would contaminate any media-bias measure. This costs about 5 cents and runs only if `RUN_SWAP_TESTS = True`.

In [ ]:
RUN_SWAP_TESTS = True
# One pass, so a name swapped IN is never swapped again (an earlier draft chained replacements and could produce "Donald Biden").
SWAP_MAP = {"Donald Trump": "Joe Biden", "Joe Biden": "Donald Trump", "Trump": "Biden", "Biden": "Trump",
            "Republicans": "Democrats", "Republican": "Democrat", "Democrats": "Republicans", "Democrat": "Republican", "GOP": "Democrats"}
SWAP_RE = re.compile(r"\b(" + "|".join(sorted(SWAP_MAP, key=len, reverse=True)) + r")\b")
def swap_names(s): return SWAP_RE.sub(lambda m: SWAP_MAP[m.group(1)], str(s))
assert swap_names("Donald Trump blames Biden and Republicans; Trump's GOP") == "Joe Biden blames Trump and Democrats; Biden's Democrats"

named = sheet_in[sheet_in.apply(lambda r: bool(SWAP_RE.search(" ".join([r.title, r.standfirst, r.text_excerpt]))), axis=1)].copy().reset_index(drop=True)
print(f"Items that name Trump, Biden or a party: {len(named)} of 300")

# DESIGN (v2). A first version put the original and swapped copy of every item in the same sheet. The labeler then returned
# identical labels for all pairs (100% tone match, against about 89% when the same item was labeled in two separate sessions),
# which shows it recognised the pairs and copied labels, so that result says nothing about name bias. In v2 every item appears
# ONCE: a random half is shown swapped, the other half unchanged. Both halves are compared with the labels from the earlier,
# separate session. The unchanged half measures plain re-labeling noise; the swapped half measures noise plus any name effect.
SWAP_SHEET, SWAP_KEY = DATA_PROC / "day4_swap_sheet_v2_BLIND.xlsx", DATA_PROC / "day4_swap_key_v2.csv"
rng_sw = np.random.default_rng(SEED)
named["version"] = np.where(rng_sw.random(len(named)) < 0.5, "swapped", "original")
shown = named.copy()
msk = shown.version == "swapped"
for c in ("title", "standfirst", "text_excerpt"): shown.loc[msk, c] = shown.loc[msk, c].map(swap_names)
shown = shown.sample(frac=1, random_state=SEED + 1).reset_index(drop=True)
shown["swap_id"] = [f"S{i + 1:03d}" for i in range(len(shown))]
swap_sheet = pd.DataFrame({"label_id": shown.swap_id, "title": shown.title, "standfirst": shown.standfirst,
                           "text_excerpt": shown.text_excerpt, "us_econ": "", "topic": "", "tone": "", "frame": "", "blame": "",
                           "confidence": "", "note": ""})
if not SWAP_SHEET.exists():
    write_sheet(swap_sheet, SWAP_SHEET)
    shown[["swap_id", "label_id", "version"]].to_csv(SWAP_KEY, index=False)
    print(f"Wrote {SWAP_SHEET.name} ({len(swap_sheet)} rows, each item once) and the separate key {SWAP_KEY.name}.")

FLIP = {"trump_gop": "biden_dems", "biden_dems": "trump_gop"}
def swap_report(lab):
    key = pd.read_csv(SWAP_KEY)
    cur = lab.rename(columns={"label_id": "swap_id"}).merge(key, on="swap_id")
    prev = ai_sample.rename(columns={"key": "label_id"})
    j = cur.merge(prev, on="label_id", suffixes=("_new", "_prev"))
    j = j[(j.ai_us_econ_new.astype(int) == 1) & (j.ai_us_econ_prev.astype(int) == 1)]
    out = {}
    for v, g in j.groupby("version"):
        exp = g.ai_blame_prev.map(lambda b: FLIP.get(b, b)) if v == "swapped" else g.ai_blame_prev
        d = g.ai_tone_new.astype(float) - g.ai_tone_prev.astype(float)
        out["swapped (noise + name effect)" if v == "swapped" else "unchanged (noise floor)"] = {
            "items": len(g), "tone identical": f"{(d == 0).mean():.0%}", "tone moved 2+ points": f"{(d.abs() >= 2).mean():.0%}",
            "mean tone change": f"{d.mean():+.2f}", "blame as expected": f"{(g.ai_blame_new == exp).mean():.0%}",
            "frame identical": f"{(g.ai_frame_new == g.ai_frame_prev).mean():.0%}"}
    return pd.DataFrame(out).T

SWAP_EXT = DATA_PROC / "day4_swap_labels_v2_EXTERNAL.csv"
if SWAP_EXT.exists():
    lab = pd.read_csv(SWAP_EXT, dtype=str).fillna("")
    assert len(lab) == len(swap_sheet) and set(lab.label_id) == set(swap_sheet.label_id), "swap label file does not match the v2 swap sheet"
    res = swap_report(lab); display(res); res.to_csv(DATA_PROC / "day4_swap_test.csv")
    print("A name effect exists only if the swapped row is clearly worse than the unchanged row on tone and on blame-as-expected.")
elif RUN_SWAP_TESTS and len(ai_sample) and OPENAI_KEY and len(named) > 0:
    # API route: every call sees ONE item, so no pairing is possible. All named items are sent swapped and compared with the earlier labels.
    sw = named.copy()
    for c in ("title", "standfirst", "text_excerpt"): sw[c] = sw[c].map(swap_names)
    sw["key"] = sw.label_id
    ai_sw = ai_label(sw, "swap", 0.30)
    if len(ai_sw):
        j = ai_sample.rename(columns={"key": "label_id"}).merge(ai_sw.rename(columns={"key": "label_id"}), on="label_id", suffixes=("_prev", "_new"))
        j = j[(j.ai_us_econ_new.astype(int) == 1) & (j.ai_us_econ_prev.astype(int) == 1)]
        d = j.ai_tone_new.astype(float) - j.ai_tone_prev.astype(float)
        exp = j.ai_blame_prev.map(lambda b: FLIP.get(b, b))
        res = pd.DataFrame({"swapped (API, independent calls)": {"items": len(j), "tone identical": f"{(d == 0).mean():.0%}",
                            "tone moved 2+ points": f"{(d.abs() >= 2).mean():.0%}", "mean tone change": f"{d.mean():+.2f}",
                            "blame as expected": f"{(j.ai_blame_new == exp).mean():.0%}", "frame identical": f"{(j.ai_frame_new == j.ai_frame_prev).mean():.0%}"}}).T
        display(res); res.to_csv(DATA_PROC / "day4_swap_test.csv")
else:
    print("Swap test not run yet: label day4_swap_sheet_v2_BLIND.xlsx in a fresh chat and save it as day4_swap_labels_v2_EXTERNAL.csv, or set OPENAI_API_KEY.")

## What happens next
* Review the agreement table and the baseline comparison. If the AI is acceptable (kappa on tone and blame at least moderate, no large tone offset, swap test passes), it is used to label a larger sample for the monthly measures in Day 5. If not, the guide is revised as **version 2 with the reason recorded**, and the AI is re-run.
* The second human pass (`day4_label_sheet_pass2_BLANK.xlsx`) gives a human-human kappa to compare with the human-AI kappa.